# Chatterbox Multilingual V3: Custom-Text TTS Web UI

Run **Chatterbox Multilingual V3** in Google Colab and use it through a browser-based [Gradio](https://gradio.app/) interface. Enter your own text, choose a supported language, choose a built-in reference voice or upload/record your own voice reference, adjust generation settings, and generate downloadable WAV audio.

The long-form batch tab can synthesize a manuscript that you have already reviewed and approved. It does not extract or approve the *Buddhist Yoga* PDF, inspect the project's approval records, or change the separate repository workflow's full-book synthesis gate.

## What it provides

- Quick free-form TTS for auditions, plus a separate full-length audiobook batch workflow.
- PDF, TXT, and Markdown manuscript ingestion, with page-range selection for PDFs and automatic visual-line reflow that preserves punctuation and repairs short wrapped Chapter/Part/Book titles.
- The language list obtained directly from the loaded Chatterbox V3 model.
- Built-in English and Hindi demo references, or your own uploaded/microphone-recorded reference audio.
- Standard and experimental longer-reference conditioning profiles, plus controls for exaggeration, CFG weight, temperature, repetition penalty, min-p, top-p, and seed.
- WAV playback and download, live logs, resumable long-form jobs, selective chunk rerendering, per-section masters, whole-book assembly, technical QA reports, and delivery bundles.
- Optional spoiler-free regional accent analysis for finished WAV/MP3 audio, with timestamped windows and no speech transcription.
- The same pinned Chatterbox source revision and verified V3 model-file approach used by the project's existing notebook.

**Runtime:** GPU is strongly recommended. CPU fallback is supported but can be very slow. The selected model files require several gigabytes of download/storage.

**Google Drive is optional:** by default, the notebook does not mount or access Drive. The model and generated files stay in temporary Colab storage, and generated WAVs can be downloaded from Gradio. To keep model snapshots and generated WAVs/metadata across runtime resets, set `USE_GOOGLE_DRIVE = True` in the setup cell and rerun from the top. The first Drive-enabled run will ask you to authorize access.

**Privacy:** the Gradio share URL is reachable over the public internet and this app has no login. Anyone who obtains the URL can use the UI, so do not share it publicly. Your uploaded voice reference is processed in the Colab runtime and is not copied into the repository or the Drive output folder. A normalized temporary copy remains under the Colab VM's `/content` directory until that runtime is terminated or reset. Generated JSON metadata includes the entered text and generation settings, so treat saved outputs as private.

**Accent scan:** the separate Accent Check tab uses the open-source AccentCL classifier to estimate broad English accent regions in overlapping audio windows. It does not transcribe the audiobook or expose sentence text. This is a diagnostic aid, not a pronunciation or accent-quality guarantee; first use downloads an additional multi-gigabyte model and a GPU is recommended.

**Not a translator:** choose the language that matches the text you entered. For best voice consistency, use a clean reference recording in that same language. The English/Hindi built-in voices are only convenient defaults; for other languages, upload a suitable reference recording.

**Text limit:** up to 12,000 characters per request. Quick TTS requests are limited to 12,000 characters. Physical line wraps are cleaned into continuous prose while blank lines remain paragraph boundaries. For a full manuscript, use the Audiobook Jobs tab: it chunks and checkpoints the book, saves each WAV, and can resume after a Colab reset when Drive persistence is enabled.

## 1. Optional Drive setup and install the pinned runtime

This uses the same Chatterbox upstream commit as the existing pronunciation notebook. Gradio is pinned to **6.31.0**. NumPy follows the pinned Chatterbox package metadata: Python 3.13 and newer use NumPy 2.x, while older Python versions use NumPy 1.x. The Colab runtime's Torch and torchaudio versions are preserved. Package installation progress is shown in the cell output.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import shutil
import json
import hashlib
import gc
import time
import re
import threading
import uuid
import urllib.request
from datetime import datetime, timezone

# Optional persistence. False means no Google Drive authorization is requested.
# Change to True before running this cell to preserve model files and generated outputs.
USE_GOOGLE_DRIVE = False

from google.colab import drive
if USE_GOOGLE_DRIVE:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/Chatterbox Playground")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print("Google Drive persistence: ON")
else:
    DRIVE_ROOT = None
    print("Google Drive persistence: OFF (temporary Colab storage only)")

CHATTERBOX_COMMIT = "5de7a54aa4e5e2baadb0182dde554908b48b85c2"

# Match the NumPy requirement in the pinned Chatterbox pyproject.toml:
# Python 3.13+ requires NumPy 2.x; Python 3.12 and older use NumPy 1.x.
NUMPY_REQUIREMENT = (
    "numpy>=2.0.0" if sys.version_info >= (3, 13) else "numpy<2.0.0"
)
print("Python:", sys.version.split()[0], "| NumPy requirement:", NUMPY_REQUIREMENT)

# Let pip show download/build progress instead of appearing frozen.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--progress-bar", "on",
        NUMPY_REQUIREMENT, "gradio==6.31.0",
    ],
    check=True,
)

import numpy as np
import torch
import torchaudio

TORCH_BEFORE = torch.__version__
TORCHAUDIO_BEFORE = torchaudio.__version__
print("Colab Python:", sys.version.split()[0])
print("Torch:", TORCH_BEFORE, "| torchaudio:", TORCHAUDIO_BEFORE)
print("NumPy:", np.__version__)

numpy_major = int(np.__version__.split(".")[0])
if sys.version_info >= (3, 13) and numpy_major < 2:
    raise RuntimeError("Python 3.13+ requires NumPy 2.x for this Chatterbox revision. Rerun the setup cell.")
if sys.version_info < (3, 13) and numpy_major >= 2:
    raise RuntimeError("Python 3.12 and older require NumPy 1.x for this Chatterbox revision. Rerun the setup cell.")

# Remove an incompatible PyPI installation if one is already present.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "chatterbox-tts"],
    check=False,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

source_spec = f"git+https://github.com/resemble-ai/chatterbox.git@{CHATTERBOX_COMMIT}"
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--progress-bar", "on",
        "--no-deps", source_spec,
    ],
    check=True,
)

dependency_specs = [
    "librosa==0.11.0",
    "s3tokenizer",
    "transformers==5.2.0",
    "diffusers==0.29.0",
    "git+https://github.com/resemble-ai/Perth.git@master",
    "conformer==0.3.2",
    "safetensors==0.5.3",
    "omegaconf",
    "soundfile",
    "huggingface_hub",
    "pypdf",
    "pypdf>=5.0,<7",
]
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--progress-bar", "on",
        *dependency_specs,
    ],
    check=True,
)

import librosa
import soundfile as sf
import gradio as gr
from safetensors import safe_open
from huggingface_hub import HfApi, snapshot_download
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

# Apply the pinned source overlay that separates narrator identity embeddings from
# optional accent/style acoustic prompts. The model checkpoint weights are unchanged.
ACCENT_FACTORIZATION_PATCH_COMMIT = "9ea38a1d6708bf632f3de185eda1b51617843b0a"
ACCENT_FACTORIZATION_PATCH_PATH = Path("/content/chatterbox_accent_factorization.py")
ACCENT_FACTORIZATION_PATCH_URL = (
    "https://raw.githubusercontent.com/privacyfreak84/colabchatterLLMthingstuff/"
    f"{ACCENT_FACTORIZATION_PATCH_COMMIT}/tools/chatterbox_accent_factorization.py"
)
urllib.request.urlretrieve(
    ACCENT_FACTORIZATION_PATCH_URL,
    ACCENT_FACTORIZATION_PATCH_PATH,
)
sys.path.insert(0, str(ACCENT_FACTORIZATION_PATCH_PATH.parent))
from chatterbox_accent_factorization import install_chatterbox_accent_factorization
ACCENT_FACTORIZATION_PATCH_STATUS = install_chatterbox_accent_factorization(
    ChatterboxMultilingualTTS
)
print("Chatterbox accent/identity patch:", ACCENT_FACTORIZATION_PATCH_STATUS)

assert torch.__version__ == TORCH_BEFORE, (
    f"Torch changed during installation: {TORCH_BEFORE} -> {torch.__version__}"
)
assert torchaudio.__version__ == TORCHAUDIO_BEFORE, (
    f"torchaudio changed during installation: {TORCHAUDIO_BEFORE} -> {torchaudio.__version__}"
)

import pypdf
print("Gradio:", gr.__version__, "| pypdf:", pypdf.__version__)
print("Pinned Chatterbox source:", CHATTERBOX_COMMIT)
print("Torch and torchaudio were preserved.")

## 2. Download or reuse the verified Multilingual V3 model files

The full model repository contains files that this notebook does not need. It downloads only the six files used by the existing project workflow, pins the download to a specific Hugging Face model revision, checks their sizes, and saves a verified copy in a dedicated Drive folder.

This model snapshot is separate from the audiobook project's existing model-snapshot folder.

In [ ]:
HF_HOME = Path("/content/chatterbox-playground-hf")
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HOME / "hub")

MODEL_REPO = "ResembleAI/chatterbox"
model_info = HfApi().model_info(MODEL_REPO, revision="main", files_metadata=True)
MODEL_REVISION = model_info.sha

MODEL_FILES = [
    "ve.pt",
    "s3gen.pt",
    "t3_mtl23ls_v3.safetensors",
    "grapheme_mtl_merged_expanded_v1.json",
    "conds.pt",
    "Cangjie5_TC.json",
]
REMOTE_SIZES = {
    item.rfilename: getattr(item, "size", None)
    for item in model_info.siblings
    if item.rfilename in MODEL_FILES
}
MINIMUM_SIZES = {
    "ve.pt": 5_000_000,
    "s3gen.pt": 1_000_000_000,
    "t3_mtl23ls_v3.safetensors": 2_000_000_000,
    "grapheme_mtl_merged_expanded_v1.json": 60_000,
    "conds.pt": 50_000,
    "Cangjie5_TC.json": 1_000_000,
}

DRIVE_MODEL_DIR = (
    DRIVE_ROOT / "model-snapshots" / f"chatterbox-v3-{MODEL_REVISION[:12]}"
    if USE_GOOGLE_DRIVE else None
)
LOCAL_MODEL_DIR = Path("/content/chatterbox-playground-model")


def check_snapshot(folder: Path) -> list[str]:
    problems = []
    for filename in MODEL_FILES:
        path = folder / filename
        if not path.is_file():
            problems.append(f"{filename}: missing")
            continue

        actual_size = path.stat().st_size
        expected_size = REMOTE_SIZES.get(filename)
        if expected_size and actual_size != expected_size:
            problems.append(
                f"{filename}: {actual_size:,} bytes; Hub reports {expected_size:,}"
            )
        elif not expected_size and actual_size < MINIMUM_SIZES[filename]:
            problems.append(f"{filename}: suspiciously small ({actual_size:,} bytes)")
    return problems


if LOCAL_MODEL_DIR.exists():
    shutil.rmtree(LOCAL_MODEL_DIR)
LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

if USE_GOOGLE_DRIVE and not check_snapshot(DRIVE_MODEL_DIR):
    print("Reusing the verified Multilingual V3 snapshot from Drive.")
    for filename in MODEL_FILES:
        shutil.copy2(DRIVE_MODEL_DIR / filename, LOCAL_MODEL_DIR / filename)
else:
    if USE_GOOGLE_DRIVE and DRIVE_MODEL_DIR.exists():
        print("The dedicated Drive snapshot is incomplete or mismatched; replacing that snapshot only.")
        shutil.rmtree(DRIVE_MODEL_DIR)

    print("Downloading the selected V3 model files. This is several gigabytes...")
    snapshot_download(
        repo_id=MODEL_REPO,
        repo_type="model",
        revision=MODEL_REVISION,
        allow_patterns=MODEL_FILES,
        local_dir=str(LOCAL_MODEL_DIR),
        force_download=True,
    )

    problems = check_snapshot(LOCAL_MODEL_DIR)
    if problems:
        raise RuntimeError("Model snapshot verification failed:\n" + "\n".join(problems))

    if USE_GOOGLE_DRIVE:
        try:
            DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
            for filename in MODEL_FILES:
                shutil.copy2(LOCAL_MODEL_DIR / filename, DRIVE_MODEL_DIR / filename)

            (DRIVE_MODEL_DIR / "snapshot_manifest.json").write_text(
                json.dumps(
                    {
                        "repository": MODEL_REPO,
                        "revision": MODEL_REVISION,
                        "files": {
                            filename: {
                                "size_bytes": (DRIVE_MODEL_DIR / filename).stat().st_size
                            }
                            for filename in MODEL_FILES
                        },
                    },
                    indent=2,
                ),
                encoding="utf-8",
            )
            print("Verified model files saved to Drive.")
        except Exception as exc:
            print("Could not persist the model snapshot to Drive:", type(exc).__name__, str(exc))
            print("The verified local snapshot is still available for this runtime.")
    else:
        print("Model snapshot is in temporary Colab storage and will need downloading again after a runtime reset.")

problems = check_snapshot(LOCAL_MODEL_DIR)
if problems:
    raise RuntimeError("Local model snapshot is incomplete:\n" + "\n".join(problems))

print("Model revision:", MODEL_REVISION)
for filename in MODEL_FILES:
    print(f"{filename}: {(LOCAL_MODEL_DIR / filename).stat().st_size / 1024**2:.1f} MiB")

## 3. Preflight and load Chatterbox Multilingual V3

This checks the two PyTorch checkpoints with safe `weights_only=True` loading, checks the safetensors header, preserves the runtime's Torch versions, and loads the explicit V3 model selector. The notebook falls back to CPU if Colab did not provide a GPU.

In [ ]:
def safe_checkpoint_preflight(filename: str, force_refresh: bool = False) -> None:
    path = LOCAL_MODEL_DIR / filename
    try:
        state = torch.load(
            path,
            map_location="cpu",
            weights_only=True,
            mmap=True,
        )
        if not isinstance(state, dict) or not state:
            raise RuntimeError(f"{filename} did not contain a non-empty state dictionary.")
        print(f"{filename}: safe checkpoint read ({len(state)} top-level entries)")
        del state
        gc.collect()
    except Exception as exc:
        if force_refresh:
            raise RuntimeError(
                f"{filename} still fails after a fresh Hub download: {type(exc).__name__}: {exc}"
            ) from exc
        print(f"{filename} failed preflight ({type(exc).__name__}: {exc}). Refreshing this file only.")
        snapshot_download(
            repo_id=MODEL_REPO,
            repo_type="model",
            revision=MODEL_REVISION,
            allow_patterns=[filename],
            local_dir=str(LOCAL_MODEL_DIR),
            force_download=True,
        )
        safe_checkpoint_preflight(filename, force_refresh=True)
        if USE_GOOGLE_DRIVE and DRIVE_MODEL_DIR.exists():
            shutil.copy2(path, DRIVE_MODEL_DIR / filename)


safe_checkpoint_preflight("ve.pt")
safe_checkpoint_preflight("s3gen.pt")

with safe_open(
    str(LOCAL_MODEL_DIR / "t3_mtl23ls_v3.safetensors"),
    framework="pt",
    device="cpu",
) as handle:
    if not list(handle.keys()):
        raise RuntimeError("The V3 T3 safetensors file is empty.")
    print("V3 T3 safetensors header: valid")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", gpu.name, "| VRAM GiB:", round(gpu.total_memory / 1024**3, 1))
else:
    print("No GPU detected. Loading on CPU; generation may be very slow.")

print(f"Loading Chatterbox Multilingual V3 on {DEVICE}...")
load_started = time.perf_counter()
model = ChatterboxMultilingualTTS.from_local(
    str(LOCAL_MODEL_DIR),
    device=DEVICE,
    t3_model="v3",
)
load_seconds = time.perf_counter() - load_started

assert int(model.sr) == 24000, f"Expected 24 kHz output, got {model.sr}."
SUPPORTED_LANGUAGES = ChatterboxMultilingualTTS.get_supported_languages()
assert isinstance(SUPPORTED_LANGUAGES, dict) and "en" in SUPPORTED_LANGUAGES

LANGUAGE_CHOICES = [
    (f"{language_name} ({language_code})", language_code)
    for language_code, language_name in sorted(SUPPORTED_LANGUAGES.items())
]

print(f"Loaded in {load_seconds:.1f} seconds.")
print(f"Device: {DEVICE} | Output sample rate: {model.sr} Hz")
print("Supported language IDs:", ", ".join(sorted(SUPPORTED_LANGUAGES)))
print("The language dropdown in the next cell is populated from this model.")

## 4. Prepare reference voices, output folders, and synthesis helpers

A reference clip is used for voice conditioning, not as text to be transcribed. The standard profile uses the model's original 6-second T3 prompt and 10-second S3Gen acoustic prompt. Two experimental profiles can use a longer speaker sample and/or extend those prompt windows. Longer windows are not the upstream-trained defaults, so compare their audio against the standard profile rather than assuming longer is better.

Your own recording is normalized to mono 24 kHz PCM. Experimental profiles can retain up to 30 seconds for the speaker-identity encoder; the reference remains temporary in the Colab runtime and is not copied to Drive.

In [ ]:
REFERENCE_DIR = Path("/content/chatterbox-playground/references")
LOCAL_OUTPUT_DIR = Path("/content/chatterbox-playground/outputs")
DRIVE_OUTPUT_DIR = (DRIVE_ROOT / "generated") if USE_GOOGLE_DRIVE else None
for folder in (REFERENCE_DIR, LOCAL_OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
if DRIVE_OUTPUT_DIR is not None:
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Three inference-time reference profiles. The first is the upstream default.
# The longer profiles are experiments, not upstream-supported/training defaults.
CONDITIONING_PROFILES = {
    "Standard (T3 6s / S3Gen 10s)": {
        "t3_prompt_seconds": 6,
        "s3gen_prompt_seconds": 10,
        "voice_encoder_max_seconds": 10,
        "experimental": False,
    },
    "Longer VoiceEncoder sample (T3 6s / S3Gen 10s / VoiceEncoder up to 30s)": {
        "t3_prompt_seconds": 6,
        "s3gen_prompt_seconds": 10,
        "voice_encoder_max_seconds": 30,
        "experimental": True,
    },
    "Extended prompts (T3 10s / S3Gen 15s / VoiceEncoder up to 30s)": {
        "t3_prompt_seconds": 10,
        "s3gen_prompt_seconds": 15,
        "voice_encoder_max_seconds": 30,
        "experimental": True,
    },
}
T3_REFERENCE_SAMPLE_RATE = 16_000
SPEECH_TOKEN_RATE = 25
MAX_REFERENCE_AUDIO_SECONDS = 30

DEMO_REFERENCE_URLS = {
    "english": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/en_f1.flac",
    "hindi": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/hi_f1.flac",
}
DEMO_REFERENCE_FILES = {
    language: REFERENCE_DIR / f"official_demo_{language}.flac"
    for language in DEMO_REFERENCE_URLS
}

for language, url in DEMO_REFERENCE_URLS.items():
    path = DEMO_REFERENCE_FILES[language]
    if not path.is_file() or path.stat().st_size < 1024:
        print(f"Downloading built-in {language} reference...")
        urllib.request.urlretrieve(url, path)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def normalize_reference_audio(
    source_path: str | Path,
    max_duration_seconds: int = 10,
) -> tuple[Path, dict]:
    """Normalize a reference to mono 24 kHz PCM, retaining the requested leading window."""
    if not 1 <= int(max_duration_seconds) <= MAX_REFERENCE_AUDIO_SECONDS:
        raise ValueError(f"Reference duration must be between 1 and {MAX_REFERENCE_AUDIO_SECONDS} seconds.")
    source = Path(source_path)
    if not source.is_file():
        raise FileNotFoundError(f"Reference audio not found: {source}")

    audio, sample_rate = librosa.load(str(source), sr=24000, mono=True)
    if audio.ndim != 1 or audio.size == 0 or not np.isfinite(audio).all():
        raise ValueError("The reference audio is empty or invalid.")
    original_seconds = audio.size / 24000
    if original_seconds < 1.0:
        raise ValueError("Reference audio must contain at least one second of usable audio.")

    audio = audio[: int(max_duration_seconds) * 24000]
    pad = (-audio.size) % 960
    if pad:
        audio = np.pad(audio, (0, pad))

    digest = hashlib.sha256(audio.tobytes()).hexdigest()
    normalized_path = REFERENCE_DIR / f"normalized_{digest[:16]}.wav"
    if not normalized_path.is_file():
        sf.write(str(normalized_path), audio, 24000, subtype="PCM_16")

    return normalized_path, {
        "source_name": source.name,
        "source_sha256": sha256_file(source),
        "normalized_sha256": sha256_file(normalized_path),
        "original_duration_seconds": round(original_seconds, 3),
        "conditioning_duration_seconds": round(audio.size / 24000, 3),
        "sample_rate": 24000,
        "maximum_reference_duration_seconds": int(max_duration_seconds),
    }


def split_text_into_chunks(text: str, max_chars: int = 420) -> list[str]:
    """Pack sentences into conservative chunks; split very long sentences at word boundaries."""
    cleaned = (
        longform.normalize_manuscript_text(text)
        if "longform" in globals()
        else text.replace("\\r\\n", "\\n").replace("\\r", "\\n").strip()
    )
    if not cleaned:
        raise ValueError("Enter some text first.")
    if len(cleaned) > MAX_TEXT_CHARACTERS:
        raise ValueError(f"Text is longer than the {MAX_TEXT_CHARACTERS:,}-character limit.")

    paragraphs = [part.strip() for part in cleaned.splitlines() if part.strip()]
    if not paragraphs:
        raise ValueError("Enter some text first.")

    chunks = []
    for paragraph in paragraphs:
        sentences = re.split(r"(?<=[.!?])\s+", paragraph)
        current = ""

        for sentence in sentences:
            sentence = sentence.strip()
            if not sentence:
                continue

            if len(sentence) > max_chars:
                if current:
                    chunks.append(current)
                    current = ""
                words = sentence.split()
                piece = ""
                for word in words:
                    if len(word) > max_chars:
                        raise ValueError(
                            f"A single word/token is longer than the {max_chars}-character chunk limit."
                        )
                    candidate = word if not piece else piece + " " + word
                    if len(candidate) <= max_chars:
                        piece = candidate
                    else:
                        chunks.append(piece)
                        piece = word
                if piece:
                    chunks.append(piece)
                continue

            candidate = sentence if not current else current + " " + sentence
            if len(candidate) <= max_chars:
                current = candidate
            else:
                if current:
                    chunks.append(current)
                current = sentence

        if current:
            chunks.append(current)

    if not chunks:
        raise ValueError("No usable text remained after splitting.")
    return chunks


MODEL_LOCK = threading.Lock()
INTER_CHUNK_PAUSE_SECONDS = 0.18
MAX_TEXT_CHARACTERS = 12000
MAX_CHUNK_CHARACTERS = 420

print("Reference voices cached.")
print("Local temporary output:", LOCAL_OUTPUT_DIR)
print("Drive output:", DRIVE_OUTPUT_DIR if DRIVE_OUTPUT_DIR is not None else "disabled")
print("Sentence-aware chunker ready.")

## 5. Build the text-to-speech function

Each request prepares the selected reference voice, generates the text chunks in order, inserts a short pause between chunks, writes a 24 kHz PCM WAV, and stores a metadata sidecar. A modest peak ceiling prevents accidental full-scale overshoots when the pieces are joined.

Only one generation request runs at a time because the model keeps voice-conditioning state in memory.

In [ ]:
def audit_text_tokenization(chunks: list[str], language_id: str) -> dict:
    """Count tokenizer [UNK] tokens before synthesis; never rewrites the user's text."""
    backend = getattr(getattr(model, "tokenizer", None), "tokenizer", None)
    if backend is None or not hasattr(backend, "token_to_id"):
        return {
            "available": False,
            "unknown_token_count": None,
            "chunks": [],
        }

    unknown_id = backend.token_to_id("[UNK]")
    if unknown_id is None:
        return {
            "available": False,
            "unknown_token_count": None,
            "chunks": [],
        }

    chunk_audits = []
    for index, chunk in enumerate(chunks, start=1):
        token_tensor = model.tokenizer.text_to_tokens(chunk, language_id=language_id)
        token_ids = token_tensor.reshape(-1).tolist()
        unknown_count = sum(int(token_id) == int(unknown_id) for token_id in token_ids)
        chunk_audits.append({
            "chunk_index": index,
            "token_count": len(token_ids),
            "unknown_token_count": unknown_count,
        })

    return {
        "available": True,
        "unknown_token_count": sum(item["unknown_token_count"] for item in chunk_audits),
        "chunks": chunk_audits,
    }


def generate_speech(
    text: str,
    language_id: str,
    reference_mode: str,
    uploaded_reference: str | None,
    uploaded_accent_reference: str | None,
    accent_prompt_start_seconds: float,
    conditioning_profile: str,
    speech_rate_value: float,
    exaggeration: float,
    cfg_weight: float,
    temperature: float,
    repetition_penalty: float,
    min_p: float,
    top_p: float,
    seed_value: float,
    progress=gr.Progress(),
):
    started = time.perf_counter()
    append_app_log(
        "quick_generation_started",
        "Started a quick TTS request.",
        details={
            "language_id": language_id,
            "conditioning_profile": conditioning_profile,
            "speech_rate": float(speech_rate_value),
        },
    )

    try:
        chunks = split_text_into_chunks(text, max_chars=MAX_CHUNK_CHARACTERS)

        if conditioning_profile not in CONDITIONING_PROFILES:
            raise ValueError(f"Unknown conditioning profile: {conditioning_profile}")
        conditioning_settings = CONDITIONING_PROFILES[conditioning_profile]
        speech_rate_value = float(speech_rate_value)
        if not speech_rate.MIN_SPEECH_RATE <= speech_rate_value <= speech_rate.MAX_SPEECH_RATE:
            raise ValueError(
                f"Speech rate must be between {speech_rate.MIN_SPEECH_RATE:.2f}x "
                f"and {speech_rate.MAX_SPEECH_RATE:.2f}x."
            )

        if language_id not in SUPPORTED_LANGUAGES:
            raise ValueError(f"Unsupported language ID: {language_id}")

        text_token_audit = audit_text_tokenization(chunks, language_id)

        if reference_mode == "Upload/record my own voice":
            if not uploaded_reference:
                raise ValueError("Upload or record a reference audio clip first.")
            reference_source = Path(uploaded_reference)
            reference_label = "user-uploaded reference"
        elif reference_mode == "Built-in English demo voice":
            reference_source = DEMO_REFERENCE_FILES["english"]
            reference_label = "Chatterbox English demo"
        elif reference_mode == "Built-in Hindi demo voice":
            reference_source = DEMO_REFERENCE_FILES["hindi"]
            reference_label = "Chatterbox Hindi demo"
        elif reference_mode == "Automatic (English/Hindi demo; English fallback)":
            if language_id == "hi":
                reference_source = DEMO_REFERENCE_FILES["hindi"]
                reference_label = "automatic Hindi demo"
            else:
                reference_source = DEMO_REFERENCE_FILES["english"]
                reference_label = "automatic English demo"
        else:
            raise ValueError(f"Unknown reference mode: {reference_mode}")

        normalized_reference, reference_info = normalize_reference_audio(
            reference_source,
            max_duration_seconds=conditioning_settings["voice_encoder_max_seconds"],
        )
        normalized_accent_reference = None
        accent_reference_info = None
        if uploaded_accent_reference:
            normalized_accent_reference, accent_reference_info = normalize_reference_audio(
                uploaded_accent_reference,
                max_duration_seconds=MAX_REFERENCE_AUDIO_SECONDS,
            )

        request_seed = int(round(seed_value))
        if request_seed < 0:
            import secrets
            request_seed = secrets.randbelow(2**31 - 1)

        generation_parameters = {
            "exaggeration": float(exaggeration),
            "cfg_weight": float(cfg_weight),
            "temperature": float(temperature),
            "repetition_penalty": float(repetition_penalty),
            "min_p": float(min_p),
            "top_p": float(top_p),
        }

        progress(0, desc="Preparing reference voice...")
        pieces = []
        chunk_records = []
        model_speech_duration_seconds = 0.0

        with MODEL_LOCK:
            # Chatterbox's upstream defaults are T3=6s and S3Gen=10s.
            # The extended profile temporarily raises those per-instance
            # limits. The Perceiver and S3Gen accept variable sequence lengths,
            # but longer-than-training prompts remain experimental.
            original_conditioning_limits = (
                model.ENC_COND_LEN,
                model.DEC_COND_LEN,
                model.t3.hp.speech_cond_prompt_len,
            )
            try:
                model.ENC_COND_LEN = int(
                    conditioning_settings["t3_prompt_seconds"] * T3_REFERENCE_SAMPLE_RATE
                )
                model.DEC_COND_LEN = int(
                    conditioning_settings["s3gen_prompt_seconds"] * int(model.sr)
                )
                model.t3.hp.speech_cond_prompt_len = int(
                    conditioning_settings["t3_prompt_seconds"] * SPEECH_TOKEN_RATE
                )
                model.prepare_conditionals(
                    str(normalized_reference),
                    exaggeration=float(exaggeration),
                    accent_prompt_path=(
                        str(normalized_accent_reference)
                        if normalized_accent_reference is not None else None
                    ),
                    accent_prompt_start_seconds=float(accent_prompt_start_seconds),
                )
            finally:
                (
                    model.ENC_COND_LEN,
                    model.DEC_COND_LEN,
                    model.t3.hp.speech_cond_prompt_len,
                ) = original_conditioning_limits

            for index, chunk in enumerate(chunks):
                chunk_seed = request_seed + index
                torch.manual_seed(chunk_seed)
                if torch.cuda.is_available():
                    torch.cuda.manual_seed(chunk_seed)
                    torch.cuda.manual_seed_all(chunk_seed)

                progress(
                    (index + 1) / len(chunks),
                    desc=f"Generating chunk {index + 1}/{len(chunks)}",
                )
                generated = model.generate(
                    chunk,
                    language_id=language_id,
                    **generation_parameters,
                )
                model_waveform = generated.detach().float().cpu().numpy().squeeze()

                if (
                    model_waveform.ndim != 1
                    or model_waveform.size == 0
                    or not np.isfinite(model_waveform).all()
                ):
                    raise RuntimeError(f"Model returned an invalid waveform for chunk {index + 1}.")

                model_duration_seconds = model_waveform.size / int(model.sr)
                model_speech_duration_seconds += model_duration_seconds
                waveform = speech_rate.adjust_speech_rate(
                    model_waveform,
                    sample_rate=int(model.sr),
                    rate=speech_rate_value,
                )
                pieces.append(waveform.astype(np.float32, copy=False))
                chunk_records.append({
                    "chunk_index": index + 1,
                    "text": chunk,
                    "model_duration_seconds": round(model_duration_seconds, 3),
                    "duration_seconds": round(waveform.size / int(model.sr), 3),
                    "speech_rate": speech_rate_value,
                    "seed": chunk_seed,
                })

                if index + 1 < len(chunks):
                    pieces.append(
                        np.zeros(
                            int(int(model.sr) * INTER_CHUNK_PAUSE_SECONDS),
                            dtype=np.float32,
                        )
                    )

        combined = np.concatenate(pieces).astype(np.float32, copy=False)
        peak_before_limiter = float(np.max(np.abs(combined)))
        limiter_gain = 1.0
        if peak_before_limiter > 0.98:
            limiter_gain = 0.98 / peak_before_limiter
            combined *= limiter_gain

        request_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
        local_wav = LOCAL_OUTPUT_DIR / f"chatterbox_{request_id}.wav"
        local_metadata = LOCAL_OUTPUT_DIR / f"chatterbox_{request_id}.json"

        sf.write(str(local_wav), combined, int(model.sr), format="WAV", subtype="PCM_16")

        metadata = {
            "request_id": request_id,
            "model": "ResembleAI/chatterbox Multilingual V3",
            "model_revision": MODEL_REVISION,
            "chatterbox_source_commit": CHATTERBOX_COMMIT,
            "device": DEVICE,
            "language_id": language_id,
            "language_name": SUPPORTED_LANGUAGES[language_id],
            "reference_label": reference_label,
            "reference": reference_info,
            "accent_reference_mode": (
                "separate_style_reference" if accent_reference_info else "same_as_narrator"
            ),
            "accent_reference": accent_reference_info,
            "accent_prompt_start_seconds": (
                float(accent_prompt_start_seconds) if accent_reference_info else 0.0
            ),
            "chatterbox_source_patch": {
                "version": "1.1.0",
                "commit": ACCENT_FACTORIZATION_PATCH_COMMIT,
                "feature": "separate-speaker-identity-and-accent-style-conditioning",
            },
            "conditioning_profile": conditioning_profile,
            "speech_rate": speech_rate_value,
            "speech_rate_processor": speech_rate.PROCESSOR_ID,
            "reference_conditioning": {
                "t3_prompt_seconds": conditioning_settings["t3_prompt_seconds"],
                "s3gen_prompt_seconds": conditioning_settings["s3gen_prompt_seconds"],
                "voice_encoder_reference_seconds": reference_info["conditioning_duration_seconds"],
                "experimental": conditioning_settings["experimental"],
            },
            "text_token_audit": text_token_audit,
            "generation_parameters": generation_parameters,
            "seed": request_seed,
            "chunk_character_limit": MAX_CHUNK_CHARACTERS,
            "inter_chunk_pause_seconds": INTER_CHUNK_PAUSE_SECONDS,
            "chunks": chunk_records,
            "audio": {
                "sample_rate": int(model.sr),
                "model_speech_duration_seconds": round(model_speech_duration_seconds, 3),
                "duration_seconds": round(combined.size / int(model.sr), 3),
                "peak_before_limiter": round(peak_before_limiter, 6),
                "limiter_gain": round(limiter_gain, 6),
                "sha256": sha256_file(local_wav),
            },
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        local_metadata.write_text(
            json.dumps(metadata, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
        )

        # Drive persistence is optional. The original user reference recording
        # is never copied to Drive.
        if USE_GOOGLE_DRIVE and DRIVE_OUTPUT_DIR is not None:
            drive_wav = DRIVE_OUTPUT_DIR / local_wav.name
            drive_metadata = DRIVE_OUTPUT_DIR / local_metadata.name
            shutil.copy2(local_wav, drive_wav)
            shutil.copy2(local_metadata, drive_metadata)
            saved_location = f"Drive: {drive_wav}"
        else:
            saved_location = f"temporary Colab storage: {local_wav}"

        elapsed = time.perf_counter() - started
        token_warning = ""
        if text_token_audit.get("available") and text_token_audit["unknown_token_count"]:
            token_warning = (
                f" WARNING: tokenizer produced {text_token_audit['unknown_token_count']} "
                "unknown [UNK] text token(s). Check for characters/scripts that do not "
                "match the selected language; use a transliteration or render that term separately."
            )

        status = (
            f"Generated {len(chunks)} chunk(s) in {elapsed:.1f}s on {DEVICE}. "
            f"Duration: {combined.size / int(model.sr):.1f}s. "
            f"Reference: {reference_label}. "
            f"Conditioning: T3 {conditioning_settings['t3_prompt_seconds']}s / "
            f"S3Gen {conditioning_settings['s3gen_prompt_seconds']}s / "
            f"VoiceEncoder {reference_info['conditioning_duration_seconds']:.1f}s. "
            f"Speech rate: {speech_rate_value:.2f}x (pitch-preserving). "
            f"{'Experimental profile. ' if conditioning_settings['experimental'] else ''}"
            f"Saved to {saved_location}. Download it before the runtime ends."
            f"{token_warning}"
        )
        append_app_log(
            "quick_generation_complete",
            "Quick TTS request completed.",
            details={
                "request_id": request_id,
                "chunks": len(chunks),
                "speech_rate": speech_rate_value,
                "duration_seconds": round(combined.size / int(model.sr), 3),
            },
        )
        return str(local_wav), str(local_wav), status

    except Exception as exc:
        import traceback
        traceback.print_exc()
        append_app_log(
            "quick_generation_failed",
            f"{type(exc).__name__}: {exc}",
            level="ERROR",
        )
        return None, None, f"Generation failed: {type(exc).__name__}: {exc}"


print("Generation function ready.")
print("Maximum text:", MAX_TEXT_CHARACTERS, "characters.")
print("Chunk size:", MAX_CHUNK_CHARACTERS, "characters.")

## 6. Long-form audiobook jobs: preview, generate, resume, and assemble

This adds a separate batch workflow for an already reviewed/approved UTF-8 manuscript. It detects common section headings, creates paragraph-aware chunks, stores each chunk's exact text, seed, settings, output hash, and status, and checkpoints after every successful chunk. Failed jobs can resume without regenerating verified chunks. It creates individual section WAVs, a streamed whole-book master WAV, a JSON manifest, readable/JSONL logs, and a ZIP delivery bundle.

**Important:** this tool does not extract or approve a PDF, compare against the project's approved manuscript tracks, or change the original repository's full-book approval gate. Use the approved text exported by the document workflow. The explicit confirmation in the UI is an additional per-job safeguard, not a replacement for manuscript review.

In [ ]:
# Long-form job support is loaded from an immutable repository commit so the
# planning/checkpoint/assembly helpers are independently testable in the repo.
import importlib.util

AUDIOBOOK_HELPER_COMMIT = "90c6070f4520cceee7fe475e16ad3485aee13989"
AUDIOBOOK_HELPER_URL = (
    "https://raw.githubusercontent.com/privacyfreak84/colabchatterLLMthingstuff/"
    f"{AUDIOBOOK_HELPER_COMMIT}/tools/longform_audiobook.py"
)
AUDIOBOOK_HELPER_DIR = Path("/content/chatterbox-playground/helpers")
AUDIOBOOK_HELPER_DIR.mkdir(parents=True, exist_ok=True)
AUDIOBOOK_HELPER_PATH = AUDIOBOOK_HELPER_DIR / "longform_audiobook.py"

with urllib.request.urlopen(AUDIOBOOK_HELPER_URL, timeout=30) as response:
    helper_bytes = response.read()
if len(helper_bytes) < 10_000:
    raise RuntimeError("The long-form helper download was incomplete.")
AUDIOBOOK_HELPER_PATH.write_bytes(helper_bytes)
helper_spec = importlib.util.spec_from_file_location(
    "chatterbox_longform_audiobook", AUDIOBOOK_HELPER_PATH
)
if helper_spec is None or helper_spec.loader is None:
    raise RuntimeError("Could not load the long-form audiobook helper.")
longform = importlib.util.module_from_spec(helper_spec)
sys.modules[helper_spec.name] = longform
helper_spec.loader.exec_module(longform)

# Repair chapter titles split across PDF visual lines without changing the stable
# audiobook-helper pin or existing resumable-job compatibility.
PDF_HEADING_REFLOW_COMMIT = "b4896c7c8ef60cd0c5ac0093971573852586edc5"
PDF_HEADING_REFLOW_URL = (
    "https://raw.githubusercontent.com/privacyfreak84/colabchatterLLMthingstuff/"
    f"{PDF_HEADING_REFLOW_COMMIT}/tools/manuscript_heading_reflow.py"
)
PDF_HEADING_REFLOW_PATH = AUDIOBOOK_HELPER_DIR / "manuscript_heading_reflow.py"
with urllib.request.urlopen(PDF_HEADING_REFLOW_URL, timeout=30) as response:
    heading_reflow_bytes = response.read()
if len(heading_reflow_bytes) < 1_000:
    raise RuntimeError("The chapter-heading cleanup helper download was incomplete.")
PDF_HEADING_REFLOW_PATH.write_bytes(heading_reflow_bytes)
heading_reflow_spec = importlib.util.spec_from_file_location(
    "chatterbox_manuscript_heading_reflow", PDF_HEADING_REFLOW_PATH
)
if heading_reflow_spec is None or heading_reflow_spec.loader is None:
    raise RuntimeError("Could not load the chapter-heading cleanup helper.")
heading_reflow = importlib.util.module_from_spec(heading_reflow_spec)
sys.modules[heading_reflow_spec.name] = heading_reflow
heading_reflow_spec.loader.exec_module(heading_reflow)
PDF_HEADING_REFLOW_STATUS = heading_reflow.install_chapter_heading_reflow(longform)
print("Wrapped chapter-heading cleanup:", PDF_HEADING_REFLOW_STATUS)

# Pitch-preserving speech-rate processor, pinned independently from the notebook.
SPEECH_RATE_HELPER_COMMIT = "cbbbe10084b5a2af6c5297a9b335e44f20d0f3c8"
SPEECH_RATE_HELPER_URL = (
    "https://raw.githubusercontent.com/privacyfreak84/colabchatterLLMthingstuff/"
    f"{SPEECH_RATE_HELPER_COMMIT}/tools/speech_rate.py"
)
SPEECH_RATE_HELPER_PATH = AUDIOBOOK_HELPER_DIR / "speech_rate.py"
with urllib.request.urlopen(SPEECH_RATE_HELPER_URL, timeout=30) as response:
    speech_rate_helper_bytes = response.read()
if len(speech_rate_helper_bytes) < 2_000:
    raise RuntimeError("The speech-rate helper download was incomplete.")
SPEECH_RATE_HELPER_PATH.write_bytes(speech_rate_helper_bytes)
speech_rate_spec = importlib.util.spec_from_file_location(
    "chatterbox_speech_rate", SPEECH_RATE_HELPER_PATH
)
if speech_rate_spec is None or speech_rate_spec.loader is None:
    raise RuntimeError("Could not load the speech-rate helper.")
speech_rate = importlib.util.module_from_spec(speech_rate_spec)
sys.modules[speech_rate_spec.name] = speech_rate
speech_rate_spec.loader.exec_module(speech_rate)
print(
    "Pitch-preserving speech rate:",
    f"{speech_rate.MIN_SPEECH_RATE:.2f}x–{speech_rate.MAX_SPEECH_RATE:.2f}x",
    "| default:", f"{speech_rate.DEFAULT_SPEECH_RATE:.2f}x",
)

JOBS_ROOT = (
    DRIVE_OUTPUT_DIR / "audiobook_jobs"
    if DRIVE_OUTPUT_DIR is not None
    else LOCAL_OUTPUT_DIR / "audiobook_jobs"
)
JOBS_ROOT.mkdir(parents=True, exist_ok=True)
APP_LOG_PATH = (
    DRIVE_OUTPUT_DIR / "app_events.jsonl"
    if DRIVE_OUTPUT_DIR is not None
    else LOCAL_OUTPUT_DIR / "app_events.jsonl"
)
APP_LOG_LOCK = threading.Lock()
MAX_BATCH_FILE_BYTES = 100 * 1024 * 1024

def append_app_log(event: str, message: str, details: dict | None = None, level: str = "INFO") -> None:
    """Append a small privacy-conscious app event; never log manuscript text or audio."""
    APP_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    entry = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "level": level.upper(),
        "event": event,
        "message": message,
        "details": details or {},
    }
    with APP_LOG_LOCK:
        with APP_LOG_PATH.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(entry, ensure_ascii=False) + "\n")
            handle.flush()

def read_audiobook_source(
    uploaded_file: str | None,
    pasted_text: str | None,
    pdf_page_start: float = 1,
    pdf_page_end: float = 0,
    text_paragraph_start: float = 1,
    text_paragraph_end: float = 0,
) -> tuple[str, str, bytes | None, dict, list[dict]]:
    pasted = pasted_text or ""
    if uploaded_file and pasted.strip():
        raise ValueError("Use either an uploaded .pdf/.txt/.md manuscript or pasted text, not both.")
    if uploaded_file:
        path = Path(uploaded_file)
        if not path.is_file():
            raise FileNotFoundError("The uploaded manuscript file is no longer available.")
        suffix = path.suffix.lower()
        if suffix == ".pdf":
            if path.stat().st_size > longform.MAX_PDF_BYTES:
                raise ValueError(
                    f"PDF exceeds {longform.MAX_PDF_BYTES // (1024 * 1024)} MiB."
                )
            pdf_bytes = path.read_bytes()
            extracted = longform.extract_pdf_manuscript(
                pdf_bytes,
                page_start=int(pdf_page_start),
                page_end=int(pdf_page_end),
            )
            report = {
                key: value
                for key, value in extracted.items()
                if key not in {"text", "page_spans"}
            }
            report["source_format"] = "PDF"
            return (
                extracted["text"],
                path.name,
                pdf_bytes,
                report,
                extracted["page_spans"],
            )
        if suffix not in {".txt", ".md"}:
            raise ValueError("Supported manuscript uploads are PDF, TXT, and Markdown (.md).")
        if path.stat().st_size > MAX_BATCH_FILE_BYTES:
            raise ValueError(f"Manuscript file exceeds {MAX_BATCH_FILE_BYTES // (1024 * 1024)} MiB.")
        try:
            raw_text = path.read_text(encoding="utf-8-sig")
        except UnicodeDecodeError as exc:
            raise ValueError("Manuscript must be UTF-8 text (.txt or .md).") from exc
        normalized = longform.normalize_manuscript_text(raw_text)
        selected_text, paragraph_range = longform.select_text_paragraph_range(
            normalized,
            paragraph_start=int(text_paragraph_start),
            paragraph_end=int(text_paragraph_end),
        )
        return selected_text, path.name, None, {
            "source_format": suffix.lstrip(".").upper(),
            "raw_characters": len(raw_text),
            "normalized_characters": len(normalized),
            "line_wrap_cleanup": True,
            **paragraph_range,
        }, []
    if pasted.strip():
        normalized = longform.normalize_manuscript_text(pasted)
        selected_text, paragraph_range = longform.select_text_paragraph_range(
            normalized,
            paragraph_start=int(text_paragraph_start),
            paragraph_end=int(text_paragraph_end),
        )
        return selected_text, "pasted_manuscript.txt", None, {
            "source_format": "pasted text",
            "raw_characters": len(pasted),
            "normalized_characters": len(normalized),
            "line_wrap_cleanup": True,
            **paragraph_range,
        }, []
    raise ValueError("Upload a PDF/TXT/Markdown manuscript or paste the approved text.")


def preview_audiobook_plan(
    uploaded_file: str | None,
    pasted_text: str | None,
    pdf_page_start: float,
    pdf_page_end: float,
    text_paragraph_start: float,
    text_paragraph_end: float,
    max_chars: float,
    speak_headings: bool,
    chunk_pause: float,
    paragraph_pause: float,
    section_pause: float,
    selected_section_ids: list[str] | None = None,
) -> tuple[str, str, object]:
    try:
        source_text, source_name, pdf_bytes, ingestion_report, source_pages = read_audiobook_source(
            uploaded_file,
            pasted_text,
            pdf_page_start,
            pdf_page_end,
            text_paragraph_start,
            text_paragraph_end,
        )
        lines = [
            f"Source: {source_name}",
            f"Source format: {ingestion_report.get('source_format', 'unknown')}",
        ]
        if ingestion_report.get("source_format") == "PDF":
            lines.extend([
                f"PDF total pages: {ingestion_report.get('total_pdf_pages', 0)}",
                f"Selected pages: {ingestion_report.get('selected_page_start')}–{ingestion_report.get('selected_page_end')}",
                f"Extracted characters: {ingestion_report.get('extracted_characters', 0):,}",
                f"PDF size: {ingestion_report.get('pdf_size_bytes', 0) / 1024**2:.1f} MiB",
                f"PDF SHA-256: {ingestion_report.get('pdf_sha256', '')}",
                f"Pages with little/no selectable text: {ingestion_report.get('low_text_pages', [])[:40] or 'none detected'}",
            ])
            for warning in ingestion_report.get("warnings", []):
                lines.append(f"WARNING: {warning}")
            lines.append(
                "PDF page selection is one-based. Choose the same first and last page to speak one page only."
            )
        else:
            lines.extend([
                f"Characters before cleanup: {ingestion_report.get('raw_characters', 0):,}",
                f"Characters after cleanup: {ingestion_report.get('normalized_characters', 0):,}",
                f"Text paragraphs selected: {ingestion_report.get('selected_paragraph_start', 1)}–{ingestion_report.get('selected_paragraph_end', ingestion_report.get('total_paragraphs', 1))} of {ingestion_report.get('total_paragraphs', 1)}",
                "TXT/Markdown/pasted text has no inherent page numbers; use the paragraph range and section selector below.",
            ])
        if not source_text.strip():
            lines.extend([
                "",
                "NO USABLE TEXT WAS EXTRACTED.",
                "This PDF may be scanned/image-only. OCR is not run by this notebook; provide a searchable PDF or an OCR-processed text copy.",
            ])
            return "\n".join(lines), "", gr.Dropdown(choices=[], value=[], multiselect=True)

        full_plan = longform.plan_audiobook_chunks(
            source_text,
            max_chars=int(max_chars),
            speak_headings=bool(speak_headings),
            source_pages=source_pages,
            chunk_pause_seconds=float(chunk_pause),
            paragraph_pause_seconds=float(paragraph_pause),
            section_pause_seconds=float(section_pause),
        )
        section_choices = [
            (
                f"{section['section_id']} | {section['title']} "
                f"({len(section['chunk_ids'])} chunks)",
                section["section_id"],
            )
            for section in full_plan["sections"]
        ]
        allowed_ids = {value for _, value in section_choices}
        selected_ids = [
            str(value).upper()
            for value in (selected_section_ids or [])
            if str(value).upper() in allowed_ids
        ]
        plan = longform.filter_plan_sections(full_plan, selected_ids)

        if selected_ids:
            selection_note = "Selected sections: " + ", ".join(selected_ids)
        else:
            selection_note = "Selected sections: all (the selector is empty)"
        lines.extend([
            f"Normalized characters: {plan['source_characters']:,} | Lines: {plan['source_lines']:,}",
            f"Sections in source/page range: {plan['all_section_count']}",
            f"Sections selected for speech: {plan['section_count']} | Chunks selected: {plan['chunk_count']}",
            selection_note,
            f"Maximum chunk size: {plan['max_chunk_characters']} characters",
            f"Normalized text SHA-256: {plan['source_sha256']}",
            "",
            "SECTION PLAN",
        ])
        for section in plan["sections"]:
            lines.append(f"{section['section_id']}  {section['title']}  ({len(section['chunk_ids'])} chunks)")
            for chunk_id in section["chunk_ids"][:2]:
                chunk = next(item for item in plan["chunks"] if item["chunk_id"] == chunk_id)
                excerpt = chunk["text"].replace("\n", " ")
                if len(excerpt) > 150:
                    excerpt = excerpt[:147] + "..."
                continuation_note = " [TTS-only continuation comma]" if chunk.get("tts_boundary_comma_added") else ""
                page_note = (
                    f" [PDF page(s) {','.join(map(str, chunk['source_pages']))}]"
                    if chunk.get("source_pages") else ""
                )
                lines.append(f"    {chunk_id}: {excerpt}{continuation_note}{page_note}")
            if len(section["chunk_ids"]) > 2:
                lines.append(f"    ... {len(section['chunk_ids']) - 2} more chunk(s)")
        lines.extend([
            "",
            "No audio has been generated. Review extracted/reflowed text, headings, sections, and chunk boundaries before starting.",
            "Single visual line breaks are joined with spaces. Blank lines remain paragraph breaks. Punctuation is retained.",
            "The original PDF and normalized text are both retained in the job folder and delivery ZIP.",
            "The section selector controls which detected sections are spoken; leave it empty to speak the entire selected page range/source.",
        ])
        text_preview = source_text[:2400]
        if len(source_text) > len(text_preview):
            text_preview += "\n\n[Preview truncated. The complete normalized text is saved in the job.]"
        selection_component = gr.Dropdown(
            choices=section_choices,
            value=selected_ids,
            multiselect=True,
            interactive=True,
        )
        return "\n".join(lines), text_preview, selection_component
    except Exception as exc:
        return (
            f"Plan error: {type(exc).__name__}: {exc}",
            "",
            gr.Dropdown(choices=[], value=[], multiselect=True),
        )

def _resolve_batch_reference(reference_mode: str, uploaded_reference: str | None, language_id: str):
    if reference_mode == "Upload/record my own voice":
        if not uploaded_reference:
            raise ValueError("Upload or record the narrator reference first.")
        source = Path(uploaded_reference)
        label = "user-uploaded reference"
    elif reference_mode == "Built-in English demo voice":
        source = DEMO_REFERENCE_FILES["english"]
        label = "Chatterbox English demo"
    elif reference_mode == "Built-in Hindi demo voice":
        source = DEMO_REFERENCE_FILES["hindi"]
        label = "Chatterbox Hindi demo"
    elif reference_mode == "Automatic (English/Hindi demo; English fallback)":
        if language_id == "hi":
            source = DEMO_REFERENCE_FILES["hindi"]
            label = "automatic Hindi demo"
        else:
            source = DEMO_REFERENCE_FILES["english"]
            label = "automatic English demo"
    else:
        raise ValueError(f"Unknown reference mode: {reference_mode}")
    return source, label

def _prepare_batch_conditionals(
    normalized_reference: Path,
    settings: dict,
    exaggeration: float,
    accent_reference_path: Path | None = None,
    accent_prompt_start_seconds: float = 0.0,
) -> None:
    original_limits = (
        model.ENC_COND_LEN,
        model.DEC_COND_LEN,
        model.t3.hp.speech_cond_prompt_len,
    )
    try:
        model.ENC_COND_LEN = int(settings["t3_prompt_seconds"] * T3_REFERENCE_SAMPLE_RATE)
        model.DEC_COND_LEN = int(settings["s3gen_prompt_seconds"] * int(model.sr))
        model.t3.hp.speech_cond_prompt_len = int(
            settings["t3_prompt_seconds"] * SPEECH_TOKEN_RATE
        )
        model.prepare_conditionals(
            str(normalized_reference),
            exaggeration=float(exaggeration),
            accent_prompt_path=(
                str(accent_reference_path) if accent_reference_path is not None else None
            ),
            accent_prompt_start_seconds=float(accent_prompt_start_seconds),
        )
    finally:
        (
            model.ENC_COND_LEN,
            model.DEC_COND_LEN,
            model.t3.hp.speech_cond_prompt_len,
        ) = original_limits

def _batch_log_view(job_dir: Path) -> str:
    return longform.read_log_tail(job_dir / "job.log", max_lines=140)

def run_audiobook_job(
    uploaded_manuscript: str | None,
    pasted_manuscript: str | None,
    pdf_page_start: float,
    pdf_page_end: float,
    text_paragraph_start: float,
    text_paragraph_end: float,
    selected_section_ids: list[str] | None,
    requested_job_id: str,
    custom_output_name: str,
    rerender_chunk_ids: list[str] | str | None,
    reroll_seed: bool,
    preserve_previous_renders: bool,
    speak_headings: bool,
    max_chunk_chars: float,
    chunk_pause: float,
    paragraph_pause: float,
    section_pause: float,
    approval_confirmed: bool,
    language_id: str,
    reference_mode: str,
    uploaded_reference: str | None,
    uploaded_accent_reference: str | None,
    accent_prompt_start_seconds: float,
    conditioning_profile: str,
    speech_rate_value: float,
    exaggeration: float,
    cfg_weight: float,
    temperature: float,
    repetition_penalty: float,
    min_p: float,
    top_p: float,
    base_seed: float,
    progress=gr.Progress(track_tqdm=False),
):
    job_dir = None
    manifest = None
    try:
        if not approval_confirmed:
            raise ValueError(
                "Confirm that this is the reviewed, approved reading copy and that you authorize this audiobook batch."
            )
        source_text, source_name, source_pdf_bytes, ingestion_report, source_pages = read_audiobook_source(
            uploaded_manuscript,
            pasted_manuscript,
            pdf_page_start,
            pdf_page_end,
            text_paragraph_start,
            text_paragraph_end,
        )
        if not source_text.strip():
            raise ValueError(
                "No usable manuscript text was extracted. Inspect the extraction preview; "
                "scanned PDFs need OCR before synthesis."
            )
        if language_id not in SUPPORTED_LANGUAGES:
            raise ValueError(f"Unsupported language ID: {language_id}")
        if conditioning_profile not in CONDITIONING_PROFILES:
            raise ValueError(f"Unknown conditioning profile: {conditioning_profile}")
        settings = CONDITIONING_PROFILES[conditioning_profile]
        speech_rate_value = float(speech_rate_value)
        if not speech_rate.MIN_SPEECH_RATE <= speech_rate_value <= speech_rate.MAX_SPEECH_RATE:
            raise ValueError(
                f"Speech rate must be between {speech_rate.MIN_SPEECH_RATE:.2f}x "
                f"and {speech_rate.MAX_SPEECH_RATE:.2f}x."
            )
        reference_source, reference_label = _resolve_batch_reference(
            reference_mode, uploaded_reference, language_id
        )
        normalized_reference, reference_info = normalize_reference_audio(
            reference_source,
            max_duration_seconds=settings["voice_encoder_max_seconds"],
        )
        normalized_accent_reference = None
        accent_reference_info = None
        if uploaded_accent_reference:
            normalized_accent_reference, accent_reference_info = normalize_reference_audio(
                uploaded_accent_reference,
                max_duration_seconds=MAX_REFERENCE_AUDIO_SECONDS,
            )
        base_seed_value = int(round(base_seed))
        full_plan = longform.plan_audiobook_chunks(
            source_text,
            max_chars=int(max_chunk_chars),
            speak_headings=bool(speak_headings),
            source_pages=source_pages,
            chunk_pause_seconds=float(chunk_pause),
            paragraph_pause_seconds=float(paragraph_pause),
            section_pause_seconds=float(section_pause),
        )
        plan = longform.filter_plan_sections(full_plan, selected_section_ids)
        generation_parameters = {
            "exaggeration": float(exaggeration),
            "cfg_weight": float(cfg_weight),
            "temperature": float(temperature),
            "repetition_penalty": float(repetition_penalty),
            "min_p": float(min_p),
            "top_p": float(top_p),
        }
        text_token_audit = audit_text_tokenization(
            [item["text"] for item in plan["chunks"]],
            language_id,
        )
        generation_config = {
            "model": "ResembleAI/chatterbox Multilingual V3",
            "model_revision": MODEL_REVISION,
            "chatterbox_source_commit": CHATTERBOX_COMMIT,
            "helper_commit": AUDIOBOOK_HELPER_COMMIT,
            "language_id": language_id,
            "language_name": SUPPORTED_LANGUAGES[language_id],
            "reference_label": reference_label,
            "reference_source_sha256": reference_info["source_sha256"],
            "reference_normalized_sha256": reference_info["normalized_sha256"],
            "reference_original_duration_seconds": reference_info["original_duration_seconds"],
            "accent_reference_mode": (
                "separate_style_reference" if accent_reference_info else "same_as_narrator"
            ),
            "accent_reference_source_sha256": (
                accent_reference_info["source_sha256"] if accent_reference_info else None
            ),
            "accent_reference_normalized_sha256": (
                accent_reference_info["normalized_sha256"] if accent_reference_info else None
            ),
            "accent_prompt_start_seconds": (
                float(accent_prompt_start_seconds) if accent_reference_info else 0.0
            ),
            "speech_rate": speech_rate_value,
            "speech_rate_processor": speech_rate.PROCESSOR_ID,
            "chatterbox_source_patch": {
                "version": "1.1.0",
                "commit": ACCENT_FACTORIZATION_PATCH_COMMIT,
                "feature": "separate-speaker-identity-and-accent-style-conditioning",
            },
            "reference_conditioning": {
                "t3_prompt_seconds": settings["t3_prompt_seconds"],
                "s3gen_prompt_seconds": settings["s3gen_prompt_seconds"],
                "voice_encoder_seconds": reference_info["conditioning_duration_seconds"],
                "accent_prompt_seconds": (
                    max(settings["t3_prompt_seconds"], settings["s3gen_prompt_seconds"])
                    if accent_reference_info else None
                ),
                "experimental": settings["experimental"],
            },
            "generation_parameters": generation_parameters,
            "text_token_audit": text_token_audit,
            "base_seed": base_seed_value,
            "seed_policy": "base_seed_plus_chunk_index" if base_seed_value >= 0 else "random_seed_per_chunk_saved_in_manifest",
            "chunk_max_characters": int(max_chunk_chars),
            "selected_section_ids": plan["selected_section_ids"],
            "selection_mode": plan["selection_mode"],
            "speak_headings": bool(speak_headings),
            "pause_settings": plan["pause_settings"],
            "sample_rate": int(model.sr),
            "source_format": ingestion_report.get("source_format"),
            "source_pdf_sha256": ingestion_report.get("pdf_sha256"),
            "pdf_page_start": ingestion_report.get("selected_page_start"),
            "pdf_page_end": ingestion_report.get("selected_page_end"),
            "text_paragraph_start": (
                None if ingestion_report.get("source_format") == "PDF"
                else int(text_paragraph_start)
            ),
            "text_paragraph_end": (
                None if ingestion_report.get("source_format") == "PDF"
                else int(text_paragraph_end)
            ),
        }

        job_dir, manifest, resumed = longform.create_or_resume_job(
            JOBS_ROOT,
            requested_job_id or None,
            source_text=source_text,
            source_name=source_name,
            plan=plan,
            generation_config=generation_config,
            base_seed=base_seed_value,
            source_pdf_bytes=source_pdf_bytes,
            pdf_extraction_report=ingestion_report if source_pdf_bytes is not None else None,
            output_name=(custom_output_name or None),
        )
        job_id = manifest["job_id"]
        if text_token_audit.get("available"):
            for saved_chunk, token_record in zip(
                manifest["chunks"], text_token_audit.get("chunks", [])
            ):
                saved_chunk["unknown_text_token_count"] = int(
                    token_record.get("unknown_token_count", 0)
                )
            longform.write_manifest(job_dir, manifest)
        if isinstance(rerender_chunk_ids, str):
            rerender_values = rerender_chunk_ids.split(",")
        elif isinstance(rerender_chunk_ids, (list, tuple, set)):
            rerender_values = list(rerender_chunk_ids)
        else:
            rerender_values = []
        rerender_tokens = [
            str(item).strip().upper()
            for item in rerender_values
            if str(item).strip()
        ]
        chunk_ids_by_section = {
            section["section_id"].upper(): list(section["chunk_ids"])
            for section in manifest.get("sections", [])
        }
        known_chunk_ids = {
            chunk["chunk_id"].upper() for chunk in manifest.get("chunks", [])
        }
        requested_rerenders = []
        unknown_rerender_ids = []
        for token in rerender_tokens:
            if token in chunk_ids_by_section:
                requested_rerenders.extend(chunk_ids_by_section[token])
            elif token in known_chunk_ids:
                requested_rerenders.append(token)
            else:
                unknown_rerender_ids.append(token)
        if unknown_rerender_ids:
            raise ValueError(
                "Unknown chunk/section ID(s): " + ", ".join(unknown_rerender_ids)
            )
        requested_rerenders = list(dict.fromkeys(requested_rerenders))
        rerendered_ids = longform.force_rerender_chunks(
            job_dir,
            manifest,
            requested_rerenders,
            new_random_seed=bool(reroll_seed),
            preserve_previous=bool(preserve_previous_renders),
        )
        if rerendered_ids:
            longform.append_job_event(
                job_dir,
                "rerender_requested",
                "Selected chunks were queued for rerender.",
                details={"chunk_ids": rerendered_ids, "new_random_seed": bool(reroll_seed), "preserve_previous": bool(preserve_previous_renders)},
            )
        longform.append_job_event(
            job_dir,
            "generation_started",
            f"{'Resuming' if resumed else 'Starting'} job {job_id} on {DEVICE}.",
            details={
                "chunk_count": len(manifest["chunks"]),
                "section_count": len(manifest["sections"]),
                "reference_sha256": reference_info["source_sha256"],
                "accent_reference_sha256": (
                    accent_reference_info["source_sha256"] if accent_reference_info else None
                ),
            },
        )
        manifest["status"] = "running"
        longform.write_manifest(job_dir, manifest)
        unknown_count = text_token_audit.get("unknown_token_count")
        token_note = (
            f"\nTokenizer [UNK] count: {unknown_count}. Review those chunks in the QA report."
            if text_token_audit.get("available") and unknown_count
            else "\nTokenizer audit found no unknown text tokens."
            if text_token_audit.get("available")
            else "\nTokenizer audit unavailable for this model wrapper."
        )
        initial_status = (
            f"Job ID: {job_id}\n"
            f"{'Resuming' if resumed else 'Created'} | {len(manifest['chunks'])} chunks | "
            f"{len(manifest['sections'])} sections | {DEVICE}\n"
            "The manifest and log are checkpointed after every completed chunk."
            f"{token_note}"
        )
        manifest_path = str(job_dir / "manifest.json")
        log_path = str(job_dir / "job.log")

        # Resolve the initial player value before the first streamed update.
        # Fresh jobs start with an empty player; resumed jobs show their latest
        # already-completed, verified chunk immediately.
        total = len(manifest["chunks"])
        finished_before = sum(
            item.get("status") == "complete" and longform.verify_chunk_wav(job_dir, item)
            for item in manifest["chunks"]
        )
        latest_audio_path = next(
            (
                str(job_dir / item["wav_relpath"])
                for item in reversed(manifest["chunks"])
                if item.get("status") == "complete"
                and longform.verify_chunk_wav(job_dir, item, sample_rate=int(model.sr))
            ),
            None,
        )

        yield initial_status, _batch_log_view(job_dir), latest_audio_path, None, manifest_path, log_path, job_id, None
        with MODEL_LOCK:
            progress(0, desc="Preparing narrator conditioning...")
            _prepare_batch_conditionals(
                normalized_reference,
                settings,
                float(exaggeration),
                accent_reference_path=normalized_accent_reference,
                accent_prompt_start_seconds=float(accent_prompt_start_seconds),
            )
            longform.append_job_event(
                job_dir,
                "reference_prepared",
                "Prepared narrator conditioning for this batch.",
                details={
                    "conditioning_profile": conditioning_profile,
                    "reference_duration_seconds": reference_info["conditioning_duration_seconds"],
                    "accent_reference_used": bool(accent_reference_info),
                    "accent_reference_sha256": (
                        accent_reference_info["source_sha256"] if accent_reference_info else None
                    ),
                    "accent_prompt_start_seconds": (
                        float(accent_prompt_start_seconds) if accent_reference_info else 0.0
                    ),
                },
            )
            for chunk in manifest["chunks"]:
                if chunk.get("status") == "complete" and longform.verify_chunk_wav(job_dir, chunk):
                    continue

                chunk_id = chunk["chunk_id"]
                seed = int(chunk["seed"])
                chunk_started = time.perf_counter()
                chunk["status"] = "generating"
                chunk.pop("error", None)
                longform.write_manifest(job_dir, manifest)
                longform.append_job_event(
                    job_dir,
                    "chunk_started",
                    f"Generating {chunk_id} ({chunk['section_title']}, paragraph {chunk['paragraph_index']}).",
                    details={
                        "chunk_index": chunk["chunk_index"],
                        "seed": seed,
                        "text_sha256": chunk["text_sha256"],
                        "characters": len(chunk["text"]),
                        "speech_rate": speech_rate_value,
                    },
                )
                progress(
                    min(0.99, (finished_before + 0.25) / max(1, total)),
                    desc=f"Generating {chunk_id} ({finished_before}/{total} complete)",
                )
                try:
                    torch.manual_seed(seed)
                    if torch.cuda.is_available():
                        torch.cuda.manual_seed(seed)
                        torch.cuda.manual_seed_all(seed)
                    generated = model.generate(
                        chunk.get("tts_text", chunk["text"]),
                        language_id=language_id,
                        **generation_parameters,
                    )
                    model_waveform = generated.detach().float().cpu().numpy().squeeze()
                    if (
                        model_waveform.ndim != 1
                        or model_waveform.size == 0
                        or not np.isfinite(model_waveform).all()
                    ):
                        raise RuntimeError("Model returned an empty, non-mono, or non-finite waveform.")
                    model_duration_seconds = model_waveform.size / int(model.sr)
                    waveform = speech_rate.adjust_speech_rate(
                        model_waveform,
                        sample_rate=int(model.sr),
                        rate=speech_rate_value,
                    )
                    peak_before_limiter = float(np.max(np.abs(waveform)))
                    limiter_gain = 1.0
                    if peak_before_limiter > 0.98:
                        limiter_gain = 0.98 / peak_before_limiter
                        waveform *= limiter_gain

                    destination = job_dir / chunk["wav_relpath"]
                    destination.parent.mkdir(parents=True, exist_ok=True)
                    temporary = destination.with_name(destination.stem + ".partial.wav")
                    sf.write(
                        str(temporary),
                        waveform.astype(np.float32, copy=False),
                        int(model.sr),
                        format="WAV",
                        subtype="PCM_16",
                    )
                    os.replace(temporary, destination)
                    latest_audio_path = str(destination)
                    pending_replacement = chunk.get("pending_replacement")
                    duration = waveform.size / int(model.sr)
                    chunk.update({
                        "status": "complete",
                        "wav_sha256": longform.sha256_file(destination),
                        "model_duration_seconds": round(model_duration_seconds, 3),
                        "duration_seconds": round(duration, 3),
                        "speech_rate": speech_rate_value,
                        "elapsed_seconds": round(time.perf_counter() - chunk_started, 3),
                        "peak_before_limiter": round(peak_before_limiter, 6),
                        "limiter_gain": round(limiter_gain, 6),
                        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
                        "device": DEVICE,
                    })
                    manifest["completed_chunks"] = sum(
                        item.get("status") == "complete" for item in manifest["chunks"]
                    )
                    chunk.pop("pending_replacement", None)
                    longform.write_manifest(job_dir, manifest)
                    finished_before += 1
                    longform.append_job_event(
                        job_dir,
                        "chunk_complete",
                        f"Saved {chunk_id}: {duration:.2f}s audio.",
                        details={
                            "chunk_index": chunk["chunk_index"],
                            "seed": seed,
                            "wav_sha256": chunk["wav_sha256"],
                            "elapsed_seconds": chunk["elapsed_seconds"],
                        },
                    )
                    progress(
                        finished_before / max(1, total),
                        desc=f"Completed {finished_before}/{total} chunks",
                    )
                    current_status = (
                        f"Job ID: {job_id}\n"
                        f"Completed {finished_before}/{total} chunks. "
                        f"Last: {chunk_id}, {duration:.1f}s, seed {seed}, "
                        f"rate {speech_rate_value:.2f}x.\n"
                        "Audio is checkpointed. A runtime interruption can be resumed with the same job ID and settings."
                    )
                    yield current_status, _batch_log_view(job_dir), latest_audio_path, None, manifest_path, log_path, job_id, None
                except Exception as exc:
                    rerender_error = f"{type(exc).__name__}: {exc}"
                    pending_replacement = chunk.pop("pending_replacement", None)
                    destination = job_dir / chunk["wav_relpath"]
                    restored_previous = False
                    if pending_replacement and destination.is_file():
                        try:
                            restored_previous = (
                                longform.sha256_file(destination)
                                == pending_replacement.get("wav_sha256")
                            )
                        except OSError:
                            restored_previous = False
                    if restored_previous:
                        for key in (
                            "seed", "wav_sha256", "duration_seconds",
                            "elapsed_seconds", "peak_before_limiter", "limiter_gain",
                            "completed_at_utc", "device",
                        ):
                            value = pending_replacement.get(key)
                            if value is not None:
                                chunk[key] = value
                        chunk["status"] = "complete"
                        chunk["last_rerender_error"] = rerender_error
                        chunk.pop("error", None)
                    else:
                        chunk["status"] = "failed"
                        chunk["error"] = rerender_error
                    manifest.setdefault("errors", []).append({
                        "chunk_id": chunk_id,
                        "error": chunk["error"],
                        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                    })
                    manifest["status"] = "paused_after_error"
                    longform.write_manifest(job_dir, manifest)
                    longform.append_job_event(
                        job_dir,
                        "chunk_failed",
                        f"{chunk_id} failed: {type(exc).__name__}: {exc}",
                        level="ERROR",
                        details={"chunk_index": chunk["chunk_index"], "seed": seed},
                    )
                    yield (
                        f"Job ID: {job_id}\nStopped at {chunk_id}. {type(exc).__name__}: {exc}\n"
                        "Completed chunks are preserved. Fix the cause and resume with the same job ID/settings.",
                        _batch_log_view(job_dir),
                        latest_audio_path,
                        None,
                        manifest_path,
                        log_path,
                        job_id,
                        None,
                    )
                    return

        manifest = longform.read_manifest(job_dir)
        manifest["status"] = "complete"
        longform.write_manifest(job_dir, manifest)
        progress(0.9, desc="Preparing technical QA report and delivery bundle...")
        qa_report = longform.build_audio_qa_report(job_dir, manifest, sample_rate=int(model.sr))
        bundle = longform.create_job_bundle(job_dir)
        append_app_log(
            "audiobook_job_complete",
            f"Completed audiobook job {job_id}.",
            details={
                "job_id": job_id,
                "chunks": len(manifest["chunks"]),
                "sections": len(manifest["sections"]),
                "speech_rate": speech_rate_value,
            },
        )
        progress(1.0, desc="Chunk WAVs and delivery bundle ready.")
        final_status = (
            f"COMPLETE: {job_id}\n"
            f"{len(manifest['chunks'])} chunks across {len(manifest['sections'])} sections.\n"
            f"Speech rate: {speech_rate_value:.2f}x.\n"
            "Each chunk WAV is saved separately. No master WAV was created automatically.\n"
            f"Technical QA: {qa_report['flagged_chunk_count']} chunk(s) flagged for listening review. "
            "See the QA report; automated checks cannot judge accent, pronunciation, or prosody. "
            "Use Listen & Assemble to play saved chunks or assemble your chosen WAVs."
        )
        yield (
            final_status,
            _batch_log_view(job_dir),
            latest_audio_path,
            str(bundle),
            str(job_dir / "manifest.json"),
            str(job_dir / "job.log"),
            job_id,
            str(job_dir / "qa_report.json"),
        )
    except Exception as exc:
        if job_dir is not None:
            try:
                longform.append_job_event(
                    job_dir,
                    "job_failed",
                    f"{type(exc).__name__}: {exc}",
                    level="ERROR",
                )
                current_manifest = longform.read_manifest(job_dir)
                current_manifest["status"] = "failed"
                longform.write_manifest(job_dir, current_manifest)
            except Exception:
                pass
            log_text = _batch_log_view(job_dir)
            manifest_file = str(job_dir / "manifest.json")
            log_file = str(job_dir / "job.log")
            job_id_for_output = job_dir.name
            job_prefix = f"Job ID: {job_dir.name}\n"
        else:
            log_text = "No job log was created."
            manifest_file = None
            log_file = None
            job_id_for_output = requested_job_id or None
            job_prefix = ""
        yield (
            f"{job_prefix}Audiobook job failed: {type(exc).__name__}: {exc}",
            log_text,
            None,
            None,
            manifest_file,
            log_file,
            job_id_for_output,
            None,
        )

def list_recent_audiobook_jobs(limit: int = 30) -> str:
    rows = []
    if not JOBS_ROOT.exists():
        return "No saved audiobook jobs found."
    for folder in JOBS_ROOT.iterdir():
        manifest_path = folder / "manifest.json"
        if not folder.is_dir() or not manifest_path.is_file():
            continue
        try:
            manifest = longform.read_manifest(folder)
            chunks = manifest.get("chunks", [])
            complete = sum(item.get("status") == "complete" for item in chunks)
            rows.append({
                "updated": manifest.get("updated_at_utc", ""),
                "job_id": manifest.get("job_id", folder.name),
                "output_name": manifest.get("output_name") or "",
                "status": manifest.get("status", "unknown"),
                "complete": complete,
                "total": len(chunks),
                "sections": len(manifest.get("sections", [])),
                "duration": (
                    manifest.get("artifacts", {}).get("whole_book_master", {})
                    .get("audio", {}).get("duration_seconds")
                ),
            })
        except Exception as exc:
            rows.append({
                "updated": "",
                "job_id": folder.name,
                "output_name": "",
                "status": f"manifest error: {type(exc).__name__}",
                "complete": 0,
                "total": 0,
                "sections": 0,
                "duration": None,
            })
    rows.sort(key=lambda item: item["updated"], reverse=True)
    if not rows:
        return "No saved audiobook jobs found."
    output = ["SAVED AUDIOBOOK JOBS (newest first)", ""]
    for item in rows[:max(1, int(limit))]:
        duration = (
            f"{item['duration'] / 60:.1f} min"
            if item["duration"] is not None else "not assembled"
        )
        output.append(
            f"{item['job_id']}"
            + (f" | files: {item['output_name']}" if item.get("output_name") else "")
            + f" | {item['status']} | "
            f"{item['complete']}/{item['total']} chunks | "
            f"{item['sections']} sections | {duration}"
        )
    output.extend(["", "Copy a Job ID into the Audiobook Jobs tab to resume, or into the log lookup field to inspect it."])
    return "\n".join(output)


def _audiobook_job_sample_rate(manifest: dict) -> int:
    """Resolve the sample rate from the saved job, without requiring the TTS model in memory."""
    settings = manifest.get("generation_config", {})
    try:
        sample_rate = int(settings.get("sample_rate", 24000))
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError("The saved job has an invalid audio sample rate.") from exc
    if not 8000 <= sample_rate <= 192000:
        raise ValueError("The saved job has an unsupported audio sample rate.")
    return sample_rate


def list_audiobook_job_choices():
    """Dropdown entries for saved audiobook jobs, newest first."""
    rows = []
    if not JOBS_ROOT.is_dir():
        return []
    for folder in JOBS_ROOT.iterdir():
        manifest_path = folder / "manifest.json"
        if not folder.is_dir() or not manifest_path.is_file():
            continue
        try:
            manifest = longform.read_manifest(folder)
            chunks = manifest.get("chunks", [])
            sample_rate = _audiobook_job_sample_rate(manifest)
            complete = sum(
                item.get("status") == "complete"
                and longform.verify_chunk_wav(folder, item, sample_rate=sample_rate)
                for item in chunks
            )
            label = (
                f"{manifest.get('job_id', folder.name)} "
                f"({complete}/{len(chunks)} chunks, {manifest.get('status', 'unknown')})"
            )
            rows.append((manifest.get("updated_at_utc", ""), label, folder.name))
        except Exception:
            continue
    rows.sort(key=lambda item: item[0], reverse=True)
    return [(label, job_id) for _, label, job_id in rows[:100]]


def _audiobook_job_root(job_id: str | None) -> Path:
    resolved = longform.safe_job_id((job_id or "").strip())
    root = (JOBS_ROOT / resolved).resolve()
    try:
        root.relative_to(JOBS_ROOT.resolve())
    except ValueError as exc:
        raise ValueError("Invalid audiobook Job ID.") from exc
    if not (root / "manifest.json").is_file():
        raise FileNotFoundError(f"No saved audiobook job named {resolved} was found.")
    return root


def refresh_rerender_chunk_choices(job_id: str | None):
    """Build a source-ordered picker for rerendering exact chunks or whole sections."""
    try:
        root = _audiobook_job_root(job_id)
        manifest = longform.read_manifest(root)
        sample_rate = _audiobook_job_sample_rate(manifest)
        choices = []
        for section in manifest.get("sections", []):
            section_id = str(section.get("section_id", "")).upper()
            if section_id:
                choices.append((
                    f"{section_id} | whole section | {section.get('title', 'Untitled section')}",
                    section_id,
                ))
        for chunk in manifest.get("chunks", []):
            chunk_id = str(chunk.get("chunk_id", "")).upper()
            state = str(chunk.get("status", "unknown"))
            audio_valid = (
                state == "complete"
                and longform.verify_chunk_wav(
                    root, chunk, sample_rate=sample_rate
                )
            )
            if audio_valid:
                state_label = "complete, verified"
            elif state == "complete":
                state_label = "complete, WAV needs repair"
            else:
                state_label = state
            section = (
                chunk.get("section_title")
                or chunk.get("section_id")
                or "Section"
            )
            excerpt = " ".join(str(chunk.get("text", "")).split())
            if len(excerpt) > 88:
                excerpt = excerpt[:85].rstrip() + "..."
            choices.append((
                f"{chunk_id} | {state_label} | {section} | {excerpt}",
                chunk_id,
            ))
        return (
            gr.Dropdown(
                choices=choices,
                value=[],
                multiselect=True,
                interactive=True,
            ),
            f"Loaded {len(manifest.get('chunks', []))} chunks from "
            f"{manifest.get('job_id', root.name)}. Select only the chunk(s) "
            "you want to regenerate.",
        )
    except Exception as exc:
        return (
            gr.Dropdown(choices=[], value=[], multiselect=True, interactive=True),
            f"Could not load chunk list: {type(exc).__name__}: {exc}",
        )


def list_audiobook_audio_choices(job_id: str | None):
    """List playable WAVs in one job, including chunks, backups, and manual assemblies."""
    if not job_id:
        return [], []
    root = _audiobook_job_root(job_id)
    manifest = longform.read_manifest(root)
    sample_rate = _audiobook_job_sample_rate(manifest)
    audio_choices = []
    seen = set()

    for path in root.rglob("*.wav"):
        if not path.is_file() or path.name.endswith(".partial.wav"):
            continue
        try:
            resolved = path.resolve()
            resolved.relative_to(root.resolve())
            info = sf.info(str(resolved))
            if info.frames <= 0 or info.channels != 1 or info.samplerate != sample_rate:
                continue
            relative = str(resolved.relative_to(root.resolve()))
            if relative in seen:
                continue
            seen.add(relative)
            duration = info.frames / info.samplerate
            label = f"{relative} · {duration:.1f}s"
            audio_choices.append((path.stat().st_mtime, label, relative))
        except (OSError, ValueError):
            continue

    audio_choices.sort(key=lambda item: item[0], reverse=True)
    playback_choices = [(label, relative) for _, label, relative in audio_choices[:250]]

    chunk_choices = []
    for chunk in manifest.get("chunks", []):
        if (
            chunk.get("status") == "complete"
            and longform.verify_chunk_wav(root, chunk, sample_rate=sample_rate)
        ):
            info = sf.info(str(root / chunk["wav_relpath"]))
            duration = info.frames / info.samplerate
            section = chunk.get("section_title") or chunk.get("section_id") or "Section"
            chunk_choices.append((
                f"{chunk['chunk_id']} · {section} · {duration:.1f}s",
                chunk["chunk_id"],
            ))
    return playback_choices, chunk_choices


def refresh_audiobook_job_choices():
    choices = list_audiobook_job_choices()
    selected_job = choices[0][1] if choices else None
    playback, chunks = list_audiobook_audio_choices(selected_job) if selected_job else ([], [])
    return (
        gr.Dropdown(choices=choices, value=selected_job),
        gr.Dropdown(choices=playback, value=playback[0][1] if playback else None),
        gr.Dropdown(choices=chunks, value=[], multiselect=True),
    )


def refresh_audiobook_audio_choices(job_id: str | None):
    playback, chunks = list_audiobook_audio_choices(job_id)
    return (
        gr.Dropdown(choices=playback, value=playback[0][1] if playback else None),
        gr.Dropdown(choices=chunks, value=[], multiselect=True),
    )


def play_saved_audiobook_audio(job_id: str | None, relative_path: str | None):
    try:
        if not relative_path:
            raise ValueError("Choose a saved WAV file first.")
        root = _audiobook_job_root(job_id)
        choices, _ = list_audiobook_audio_choices(job_id)
        allowed = {value for _, value in choices}
        if relative_path not in allowed:
            raise ValueError("That WAV is not part of the selected saved audiobook job.")
        path = longform._safe_relative_path(root, relative_path)
        return str(path), f"Playing saved audio: {relative_path}"
    except Exception as exc:
        return None, f"Could not load saved audio: {type(exc).__name__}: {exc}"


def assemble_audiobook_wavs(
    job_id: str | None,
    selected_chunk_ids: list[str] | None,
    include_all_completed: bool,
    output_name: str | None,
    progress=gr.Progress(track_tqdm=False),
):
    try:
        root = _audiobook_job_root(job_id)
        manifest = longform.read_manifest(root)
        sample_rate = _audiobook_job_sample_rate(manifest)
        progress(0.15, desc="Checking selected WAV files...")
        record = longform.assemble_selected_chunk_wavs(
            root,
            manifest,
            selected_chunk_ids,
            include_all_completed=bool(include_all_completed),
            sample_rate=sample_rate,
            output_name=output_name,
        )
        bundle = None
        bundle_note = (
            "The assembled WAV is saved in this job and included in the refreshed delivery ZIP."
        )
        if manifest.get("status") == "complete":
            progress(0.8, desc="Updating delivery bundle...")
            bundle = longform.create_job_bundle(root)
        else:
            bundle_note = (
                "The assembled WAV is saved in this job. The job still has unfinished chunks, "
                "so its complete delivery ZIP will be available after generation finishes."
            )
        progress(1.0, desc="Selected WAVs assembled.")
        playback, chunks = list_audiobook_audio_choices(job_id)
        output_path = str(root / record["relpath"])
        skipped = len(manifest.get("chunks", [])) - record["chunk_count"]
        selection_label = (
            "all completed chunks"
            if record["include_all_completed"]
            else ", ".join(record["selected_chunk_ids"])
        )
        status = (
            f"Created {Path(record['relpath']).name}\n"
            f"Included {record['chunk_count']} chunk(s); {skipped} other chunk(s) not included.\n"
            f"Duration: {record['audio']['duration_seconds'] / 60:.2f} minutes.\n"
            f"Source selection: {selection_label}.\n"
            f"{bundle_note}"
        )
        return (
            output_path,
            output_path,
            status,
            str(bundle) if bundle else None,
            gr.Dropdown(choices=playback, value=record["relpath"]),
        )
    except Exception as exc:
        return (
            None,
            None,
            f"Assembly failed: {type(exc).__name__}: {exc}",
            None,
            gr.Dropdown(choices=[], value=None),
        )


def load_activity_logs(job_id: str | None):
    candidate = (job_id or "").strip()
    if candidate:
        try:
            resolved = longform.safe_job_id(candidate)
            root = JOBS_ROOT / resolved
            if not root.is_dir():
                return f"No audiobook job named {resolved} was found.", None, None, None
            log_path = root / "job.log"
            manifest_path = root / "manifest.json"
            qa_path = root / "qa_report.json"
            return (
                longform.read_log_tail(log_path, max_lines=250),
                str(manifest_path) if manifest_path.is_file() else None,
                str(log_path) if log_path.is_file() else None,
                str(qa_path) if qa_path.is_file() else None,
            )
        except Exception as exc:
            return f"Log lookup failed: {type(exc).__name__}: {exc}", None, None, None
    return (
        longform.read_log_tail(APP_LOG_PATH, max_lines=250),
        None,
        str(APP_LOG_PATH) if APP_LOG_PATH.is_file() else None,
        None,
    )



# Optional regional-accent analysis helper. This model is downloaded only when the
# user opens Accent Check and starts a scan, never during ordinary TTS setup.
ACCENT_ANALYSIS_HELPER_COMMIT = "64d57bab6cb889b377dd44eb346bdd98798d942a"
ACCENT_ANALYSIS_HELPER_URL = (
    "https://raw.githubusercontent.com/privacyfreak84/colabchatterLLMthingstuff/"
    f"{ACCENT_ANALYSIS_HELPER_COMMIT}/tools/accent_analysis.py"
)
ACCENT_ANALYSIS_HELPER_PATH = AUDIOBOOK_HELPER_DIR / "accent_analysis.py"
with urllib.request.urlopen(ACCENT_ANALYSIS_HELPER_URL, timeout=30) as response:
    accent_helper_bytes = response.read()
if len(accent_helper_bytes) < 8_000:
    raise RuntimeError("The accent-analysis helper download was incomplete.")
ACCENT_ANALYSIS_HELPER_PATH.write_bytes(accent_helper_bytes)
accent_spec = importlib.util.spec_from_file_location(
    "chatterbox_accent_analysis", ACCENT_ANALYSIS_HELPER_PATH
)
if accent_spec is None or accent_spec.loader is None:
    raise RuntimeError("Could not load the accent-analysis helper.")
accent_analysis = importlib.util.module_from_spec(accent_spec)
sys.modules[accent_spec.name] = accent_analysis
accent_spec.loader.exec_module(accent_analysis)

ACCENT_CACHE_DIR = (
    DRIVE_ROOT / "accent-model-cache"
    if DRIVE_ROOT is not None
    else LOCAL_OUTPUT_DIR / "accent-model-cache"
)
ACCENT_REPO_DIR = Path("/content/chatterbox-playground/accentcl-source")
ACCENT_REPORT_DIR = (
    DRIVE_OUTPUT_DIR / "accent-reports"
    if DRIVE_OUTPUT_DIR is not None
    else LOCAL_OUTPUT_DIR / "accent-reports"
)


def list_saved_accent_audio_choices():
    choices = []
    seen = set()
    if JOBS_ROOT.is_dir():
        for pattern in (
            "*/whole_book_master.wav",
            "*/selected_sections_master.wav",
            "*/*_whole_book_master.wav",
            "*/*_selected_sections_master.wav",
            "*/chunks/*.wav",
            "*/sections/*.wav",
            "*/assemblies/*.wav",
        ):
            for path in JOBS_ROOT.glob(pattern):
                resolved = str(path.resolve())
                if resolved not in seen and path.is_file():
                    seen.add(resolved)
                    relative = path.relative_to(JOBS_ROOT)
                    label = f"{relative.parts[0]} / {' / '.join(relative.parts[1:])}"
                    choices.append((label, resolved))
    for root in (LOCAL_OUTPUT_DIR, DRIVE_OUTPUT_DIR):
        if root is None or not Path(root).is_dir():
            continue
        for path in list(Path(root).glob("*.wav")) + list(Path(root).glob("*.mp3")):
            resolved = str(path.resolve())
            if resolved not in seen and path.is_file():
                seen.add(resolved)
                choices.append((path.name, resolved))
    choices.sort(key=lambda item: Path(item[1]).stat().st_mtime, reverse=True)
    return choices[:150]


def refresh_accent_audio_choices():
    choices = list_saved_accent_audio_choices()
    return gr.Dropdown(
        choices=choices,
        value=choices[0][1] if choices else None,
    )


def run_accent_analysis(existing_audio_path, uploaded_audio_path, progress=gr.Progress()):
    global model
    selected = str(existing_audio_path).strip() if existing_audio_path else ""
    uploaded = str(uploaded_audio_path).strip() if uploaded_audio_path else ""
    if selected and uploaded:
        raise gr.Error("Choose either an existing audiobook output or an uploaded WAV/MP3, not both.")
    audio_path = uploaded or selected
    if not audio_path:
        raise gr.Error("Select an existing master/section file or upload a WAV/MP3.")

    accent_model = None
    tts_was_unloaded = False
    tts_device = str(model.device)
    result_markdown = None
    table_rows = None
    json_report_path = None
    csv_report_path = None
    analysis_error = None

    try:
        progress(0.01, desc="Freeing TTS memory before loading the accent model...")
        del model
        tts_was_unloaded = True
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        device = "cuda" if torch.cuda.is_available() else "cpu"
        accent_model = accent_analysis.load_accentcl_model(
            cache_dir=ACCENT_CACHE_DIR,
            repo_dir=ACCENT_REPO_DIR,
            device=device,
            progress_callback=lambda message: progress(0.08, desc=message),
        )
        progress(0.12, desc="Scanning the audio in overlapping 15-second windows...")
        def update_accent_progress(done, total, start_seconds, end_seconds):
            if total:
                fraction = min(1.0, done / max(1, total))
                progress(0.12 + 0.78 * fraction, desc=f"Analysing {start_seconds:.0f} to {end_seconds:.0f} seconds")
            else:
                progress(0.15, desc=f"Analysing {start_seconds:.0f} to {end_seconds:.0f} seconds")

        report = accent_analysis.analyse_audio_file(
            audio_path,
            accent_model,
            window_seconds=15.0,
            hop_seconds=10.0,
            progress_callback=update_accent_progress,
        )
        json_report_path, csv_report_path = accent_analysis.write_accent_report(
            report, ACCENT_REPORT_DIR
        )
        result_markdown = accent_analysis.format_report_markdown(report)
        table_rows = [
            [
                row["start"], row["end"], row["accent_display"],
                f"{row['score']:.3f}", row["runner_up_display"],
                f"{row['runner_up_score']:.3f}", f"{row['margin']:.3f}",
            ]
            for row in report["segments"]
        ]
        append_app_log(
            "accent_analysis_complete",
            "Completed a spoiler-free regional accent scan.",
            details={
                "audio_name": Path(audio_path).name,
                "window_count": report["analysed_window_count"],
                "dominant_accent": report["dominant_accent"],
            },
        )
    except Exception as exc:
        analysis_error = exc
    finally:
        if accent_model is not None:
            del accent_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if tts_was_unloaded:
            progress(0.94, desc="Restoring Chatterbox so TTS can continue...")
            try:
                model = ChatterboxMultilingualTTS.from_local(
                    LOCAL_MODEL_DIR,
                    device=tts_device,
                    t3_model="v3",
                )
            except Exception as restore_error:
                raise RuntimeError(
                    "The accent scan ended, but Chatterbox could not be reloaded. "
                    "Rerun the model-loading cell before generating more audio. "
                    f"Restore error: {type(restore_error).__name__}: {restore_error}"
                ) from restore_error

    if analysis_error is not None:
        raise gr.Error(
            f"Accent analysis failed: {type(analysis_error).__name__}: {analysis_error}"
        ) from analysis_error
    progress(1.0, desc="Accent scan complete. Chatterbox has been restored.")
    return result_markdown, table_rows, json_report_path, csv_report_path


print("Long-form audiobook helpers loaded.")
print("Audiobook job root:", JOBS_ROOT)
print("App activity log:", APP_LOG_PATH)


## 7. Launch the Gradio web UI

The app has three work areas: quick one-off TTS, resumable long-form audiobook jobs, and activity/job logs. The temporary Gradio share URL has no login authentication. Anyone who obtains it can use the interface, so keep it private and stop the server when finished.

For long books, upload the reviewed UTF-8 manuscript as a .txt or .md file, preview the section/chunk plan, then confirm approval before starting. The batch job persists to Drive when Drive persistence is enabled; otherwise it lives in temporary Colab storage.

In [ ]:
REFERENCE_MODE_CHOICES = [
    "Automatic (English/Hindi demo; English fallback)",
    "Built-in English demo voice",
    "Built-in Hindi demo voice",
    "Upload/record my own voice",
]

with gr.Blocks(
    title="Chatterbox Multilingual V3 TTS and Audiobook Studio",
    analytics_enabled=False,
) as demo:
    gr.Markdown(
        """
        # Chatterbox Multilingual V3
        ### Voice playground and resumable audiobook studio

        Quick TTS is for auditions and short passages. Audiobook Jobs is for a reviewed
        manuscript: it saves every chunk, can resume after a Colab interruption, and assembles
        section masters and a whole-book WAV. Logs are available in the third tab.
        """
    )

    with gr.Tabs():
        with gr.Tab("Quick TTS"):
            with gr.Row():
                with gr.Column(scale=3):
                    text_input = gr.Textbox(
                        label="Text to speak",
                        placeholder="Type or paste your text here...",
                        lines=12,
                        max_lines=24,
                    )
                    language_input = gr.Dropdown(
                        choices=LANGUAGE_CHOICES,
                        value="en",
                        label="Text language",
                        interactive=True,
                    )
                    reference_mode_input = gr.Radio(
                        choices=REFERENCE_MODE_CHOICES,
                        value=REFERENCE_MODE_CHOICES[0],
                        label="Voice reference",
                    )
                    reference_audio_input = gr.Audio(
                        label="Upload or record narrator identity reference (used only in upload mode)",
                        sources=["upload", "microphone"],
                        type="filepath",
                        format="wav",
                    )
                    accent_reference_audio_input = gr.Audio(
                        label="Optional accent/style guide (for American English, use a clear US-English sample)",
                        sources=["upload", "microphone"],
                        type="filepath",
                        format="wav",
                    )
                    accent_prompt_start_seconds_input = gr.Number(
                        value=0,
                        precision=1,
                        label="Accent/style guide segment start (seconds; 6 uses the later part of a 19.6s recording)",
                    )
                    conditioning_profile_input = gr.Dropdown(
                        choices=list(CONDITIONING_PROFILES),
                        value="Standard (T3 6s / S3Gen 10s)",
                        label="Reference conditioning profile",
                        interactive=True,
                    )
                    speech_rate_input = gr.Slider(
                        minimum=speech_rate.MIN_SPEECH_RATE,
                        maximum=speech_rate.MAX_SPEECH_RATE,
                        value=speech_rate.DEFAULT_SPEECH_RATE,
                        step=0.05,
                        label="Narration pace (0.85x = slower; 1.00x = original)",
                    )
                    gr.Markdown(
                        "Pitch-preserving time-stretching is applied after each generated chunk. "
                        "It does not lower the voice pitch or add reverb/echo. The pauses inserted "
                        "between chunks remain at their configured length."
                    )
                    gr.Markdown(
                        "Standard is the upstream default. The longer VoiceEncoder-sample profile "
                        "uses up to 30s for the T3 VoiceEncoder identity embedding but keeps the "
                        "original 6s/10s prompt windows. The extended profile experiments with a "
                        "10s T3 prompt and 15s S3Gen prompt; this can increase memory/time and may "
                        "reduce quality because it exceeds the model's usual conditioning lengths."
                    )
                    gr.Markdown(
                        "Tip: a clean 8–10 second sample of one speaker works well. "
                        "For languages other than English or Hindi, upload a reference in the target language."
                    )

                with gr.Column(scale=2):
                    with gr.Accordion("Generation settings", open=True):
                        exaggeration_input = gr.Slider(
                            minimum=0.0, maximum=1.0, value=0.5, step=0.05,
                            label="Exaggeration",
                        )
                        cfg_input = gr.Slider(
                            minimum=0.0, maximum=1.0, value=0.5, step=0.05,
                            label="CFG weight",
                        )
                        temperature_input = gr.Slider(
                            minimum=0.05, maximum=2.0, value=0.7, step=0.05,
                            label="Temperature",
                        )
                        repetition_input = gr.Slider(
                            minimum=1.0, maximum=2.0, value=1.2, step=0.05,
                            label="Repetition penalty",
                        )
                        min_p_input = gr.Slider(
                            minimum=0.0, maximum=1.0, value=0.05, step=0.01,
                            label="Min-p",
                        )
                        top_p_input = gr.Slider(
                            minimum=0.0, maximum=1.0, value=0.95, step=0.01,
                            label="Top-p",
                        )
                        seed_input = gr.Number(
                            value=-1,
                            precision=0,
                            label="Seed (-1 = random)",
                        )
                    gr.Markdown(
                        "**Repetition-heavy passages:** try **Temperature 0.55** and "
                        "**Repetition penalty 1.35 together**. This stopped one reproduced "
                        "repetition issue; it is a targeted starting point, not a guarantee."
                    )
                    generate_button = gr.Button("Generate speech", variant="primary")

            audio_output = gr.Audio(label="Generated audio", type="filepath")
            file_output = gr.File(label="Download WAV")
            status_output = gr.Textbox(label="Status", interactive=False, lines=3)
            generate_button.click(
                fn=generate_speech,
                inputs=[
                    text_input,
                    language_input,
                    reference_mode_input,
                    reference_audio_input,
                    accent_reference_audio_input,
                    accent_prompt_start_seconds_input,
                    conditioning_profile_input,
                    speech_rate_input,
                    exaggeration_input,
                    cfg_input,
                    temperature_input,
                    repetition_input,
                    min_p_input,
                    top_p_input,
                    seed_input,
                ],
                outputs=[audio_output, file_output, status_output],
                api_name=False,
            )

        with gr.Tab("Audiobook Jobs"):
            gr.Markdown(
                """
                ### Audiobook workspace

                Work through the tabs in order: prepare and preview the manuscript, generate
                checkpointed chunk WAVs, then listen back or assemble only the WAVs you choose.
                Nothing is combined into a master automatically.
                """
            )
            with gr.Tabs():
                with gr.Tab("Create / Resume"):
                    with gr.Accordion("1. Manuscript and reading range", open=True):
                        batch_file_input = gr.File(
                            label="Upload manuscript (PDF, TXT or Markdown)",
                            file_types=[".pdf", ".txt", ".md"],
                            type="filepath",
                        )
                        batch_text_input = gr.Textbox(
                            label="Or paste manuscript text",
                            placeholder="Leave blank when uploading a file.",
                            lines=6,
                            max_lines=12,
                        )
                        gr.Markdown(
                            "Use either an uploaded file or pasted text. Searchable PDFs are supported; "
                            "scanned/image-only PDFs need OCR before upload."
                        )
                        with gr.Row():
                            pdf_page_start_input = gr.Number(
                                value=1,
                                precision=0,
                                label="PDF: first page",
                            )
                            pdf_page_end_input = gr.Number(
                                value=0,
                                precision=0,
                                label="PDF: last page (0 = final page)",
                            )
                        with gr.Row():
                            text_paragraph_start_input = gr.Number(
                                value=1,
                                precision=0,
                                label="TXT/MD: first paragraph",
                            )
                            text_paragraph_end_input = gr.Number(
                                value=0,
                                precision=0,
                                label="TXT/MD: last paragraph (0 = final paragraph)",
                            )
                        gr.Markdown(
                            "PDF ranges are one-based and inclusive. TXT/Markdown use blank-line "
                            "paragraphs because those formats do not have page numbers."
                        )

                    with gr.Accordion("2. Preview and choose what to speak", open=True):
                        speak_headings_input = gr.Checkbox(
                            value=True,
                            label="Speak detected section headings",
                        )
                        batch_chunk_chars_input = gr.Slider(
                            minimum=160,
                            maximum=520,
                            value=320,
                            step=20,
                            label="Maximum characters per speech chunk",
                        )
                        section_selection_input = gr.Dropdown(
                            choices=[],
                            value=[],
                            multiselect=True,
                            label="Sections to speak (empty selection = all)",
                            interactive=True,
                        )
                        with gr.Accordion("Pauses between chunks and paragraphs", open=False):
                            batch_chunk_pause_input = gr.Slider(
                                minimum=0.0,
                                maximum=0.8,
                                value=0.18,
                                step=0.02,
                                label="Within-paragraph chunk pause (seconds)",
                            )
                            batch_paragraph_pause_input = gr.Slider(
                                minimum=0.0,
                                maximum=1.5,
                                value=0.45,
                                step=0.05,
                                label="Paragraph pause (seconds)",
                            )
                            batch_section_pause_input = gr.Slider(
                                minimum=0.0,
                                maximum=3.0,
                                value=1.0,
                                step=0.1,
                                label="Section/chapter pause (seconds)",
                            )
                        preview_button = gr.Button("Preview manuscript and chunk plan")
                        plan_preview_output = gr.Textbox(
                            label="Reading plan",
                            lines=12,
                            max_lines=20,
                            interactive=False,
                        )
                        cleaned_text_preview_output = gr.Textbox(
                            label="Cleaned text preview",
                            lines=7,
                            max_lines=12,
                            interactive=False,
                        )
                        gr.Markdown(
                            "Previewing does not generate audio. Single visual line breaks are reflowed; "
                            "blank lines remain paragraph boundaries and punctuation is preserved."
                        )

                    with gr.Accordion("3. Narrator, accent guide and pace", open=True):
                        batch_language_input = gr.Dropdown(
                            choices=LANGUAGE_CHOICES,
                            value="en",
                            label="Manuscript language",
                            interactive=True,
                        )
                        batch_reference_mode_input = gr.Radio(
                            choices=REFERENCE_MODE_CHOICES,
                            value="Upload/record my own voice",
                            label="Narrator voice",
                        )
                        batch_reference_audio_input = gr.Audio(
                            label="Narrator identity recording",
                            sources=["upload", "microphone"],
                            type="filepath",
                            format="wav",
                        )
                        batch_accent_reference_audio_input = gr.Audio(
                            label="Optional same-language accent/style guide",
                            sources=["upload", "microphone"],
                            type="filepath",
                            format="wav",
                        )
                        batch_accent_prompt_start_seconds_input = gr.Number(
                            value=0,
                            precision=1,
                            label="Guide start offset (seconds)",
                        )
                        gr.Markdown(
                            "The narrator recording supplies speaker identity. The optional guide "
                            "steers acoustic style. You can upload the same narrator recording in both "
                            "fields and use a later offset, such as 6 seconds."
                        )
                        batch_conditioning_profile_input = gr.Dropdown(
                            choices=list(CONDITIONING_PROFILES),
                            value="Standard (T3 6s / S3Gen 10s)",
                            label="Reference conditioning profile",
                            interactive=True,
                        )
                        batch_speech_rate_input = gr.Slider(
                            minimum=speech_rate.MIN_SPEECH_RATE,
                            maximum=speech_rate.MAX_SPEECH_RATE,
                            value=speech_rate.DEFAULT_SPEECH_RATE,
                            step=0.05,
                            label="Narration pace (0.85x = slower; 1.00x = original)",
                        )
                        gr.Markdown(
                            "Pitch-preserving time-stretching is applied to each speech chunk. "
                            "It keeps the sample rate and pitch, adds no echo, and leaves your "
                            "configured pauses unchanged."
                        )

                    with gr.Accordion("4. Advanced voice settings", open=False):
                        batch_exaggeration_input = gr.Slider(
                            minimum=0.0,
                            maximum=1.0,
                            value=0.5,
                            step=0.05,
                            label="Exaggeration",
                        )
                        batch_cfg_input = gr.Slider(
                            minimum=0.0,
                            maximum=1.0,
                            value=0.5,
                            step=0.05,
                            label="CFG weight",
                        )
                        batch_temperature_input = gr.Slider(
                            minimum=0.05,
                            maximum=2.0,
                            value=0.55,
                            step=0.05,
                            label="Temperature",
                        )
                        batch_repetition_input = gr.Slider(
                            minimum=1.0,
                            maximum=2.0,
                            value=1.35,
                            step=0.05,
                            label="Repetition penalty",
                        )
                        batch_min_p_input = gr.Slider(
                            minimum=0.0,
                            maximum=1.0,
                            value=0.05,
                            step=0.01,
                            label="Min-p",
                        )
                        batch_top_p_input = gr.Slider(
                            minimum=0.0,
                            maximum=1.0,
                            value=0.95,
                            step=0.01,
                            label="Top-p",
                        )
                        batch_seed_input = gr.Number(
                            value=-1,
                            precision=0,
                            label="Base seed (-1 = choose and save random seeds)",
                        )

                    with gr.Accordion("5. Job identity and repair", open=False):
                        batch_job_id_input = gr.Textbox(
                            label="Job ID (blank = new job; reuse to resume)",
                            placeholder="Example: buddhist-yoga-chapter-01",
                        )
                        batch_output_name_input = gr.Textbox(
                            label="Optional output filename prefix",
                            placeholder="Example: Buddhist Yoga",
                        )
                        gr.Markdown(
                            "#### Repair individual chunks or a whole section"
                        )
                        batch_rerender_refresh_button = gr.Button(
                            "Load chunks from this Job ID"
                        )
                        batch_rerender_ids_input = gr.Dropdown(
                            choices=[],
                            value=[],
                            multiselect=True,
                            label="Select exact chunk(s) or section(s) to rerender",
                            interactive=True,
                        )
                        batch_rerender_choices_status = gr.Markdown(
                            "Enter an existing Job ID above, then load its chunk list. "
                            "Each choice shows the chunk ID, saved status, section, and a short text excerpt."
                        )
                        batch_reroll_seed_input = gr.Checkbox(
                            value=True,
                            label="Use fresh random seeds for rerendered chunks",
                        )
                        batch_keep_previous_renders_input = gr.Checkbox(
                            value=False,
                            label="Keep replaced WAVs as backups",
                        )
                        gr.Markdown(
                            "For a targeted repair, enter the original Job ID, load its chunks, and "
                            "select only the bad chunk (for example C00007), or select a section ID "
                            "to rerender that entire section. Keep the same manuscript, narrator/style "
                            "references, and generation settings, then click Generate / resume. "
                            "Only the selected chunks are regenerated; other verified chunks are skipped. "
                            "Enable the backup option to retain the old WAV in revisions/. "
                            "To resume after a Colab reset, enable Google Drive persistence. "
                            "A different narration rate requires a new Job ID."
                        )

                    approval_checkbox = gr.Checkbox(
                        value=False,
                        label="I reviewed and approved this exact reading copy and authorize this batch.",
                    )
                    run_batch_button = gr.Button(
                        "Generate / resume audiobook",
                        variant="primary",
                        size="lg",
                    )
                    batch_status_output = gr.Textbox(
                        label="Generation status",
                        lines=4,
                        interactive=False,
                    )
                    batch_latest_audio_output = gr.Audio(
                        label="Latest spoken chunk (updates as each chunk finishes)",
                        type="filepath",
                    )
                    batch_live_log_output = gr.Textbox(
                        label="Live job log",
                        lines=8,
                        max_lines=14,
                        interactive=False,
                    )
                    with gr.Accordion("Job files and technical report", open=False):
                        batch_bundle_output = gr.File(label="Delivery bundle ZIP (contains individual chunk WAVs)")
                        batch_manifest_output = gr.File(label="Job manifest JSON")
                        batch_log_file_output = gr.File(label="Job log TXT")
                        batch_qa_output = gr.File(label="Technical audio QA report JSON")

                    batch_rerender_refresh_button.click(
                        fn=refresh_rerender_chunk_choices,
                        inputs=[batch_job_id_input],
                        outputs=[
                            batch_rerender_ids_input,
                            batch_rerender_choices_status,
                        ],
                        api_name=False,
                    )
                    preview_button.click(
                        fn=preview_audiobook_plan,
                        inputs=[
                            batch_file_input,
                            batch_text_input,
                            pdf_page_start_input,
                            pdf_page_end_input,
                            text_paragraph_start_input,
                            text_paragraph_end_input,
                            batch_chunk_chars_input,
                            speak_headings_input,
                            batch_chunk_pause_input,
                            batch_paragraph_pause_input,
                            batch_section_pause_input,
                            section_selection_input,
                        ],
                        outputs=[
                            plan_preview_output,
                            cleaned_text_preview_output,
                            section_selection_input,
                        ],
                        api_name=False,
                    )
                    run_batch_button.click(
                        fn=run_audiobook_job,
                        inputs=[
                            batch_file_input,
                            batch_text_input,
                            pdf_page_start_input,
                            pdf_page_end_input,
                            text_paragraph_start_input,
                            text_paragraph_end_input,
                            section_selection_input,
                            batch_job_id_input,
                            batch_output_name_input,
                            batch_rerender_ids_input,
                            batch_reroll_seed_input,
                            batch_keep_previous_renders_input,
                            speak_headings_input,
                            batch_chunk_chars_input,
                            batch_chunk_pause_input,
                            batch_paragraph_pause_input,
                            batch_section_pause_input,
                            approval_checkbox,
                            batch_language_input,
                            batch_reference_mode_input,
                            batch_reference_audio_input,
                            batch_accent_reference_audio_input,
                            batch_accent_prompt_start_seconds_input,
                            batch_conditioning_profile_input,
                            batch_speech_rate_input,
                            batch_exaggeration_input,
                            batch_cfg_input,
                            batch_temperature_input,
                            batch_repetition_input,
                            batch_min_p_input,
                            batch_top_p_input,
                            batch_seed_input,
                        ],
                        outputs=[
                            batch_status_output,
                            batch_live_log_output,
                            batch_latest_audio_output,
                            batch_bundle_output,
                            batch_manifest_output,
                            batch_log_file_output,
                            batch_job_id_input,
                            batch_qa_output,
                        ],
                        api_name=False,
                    )

                with gr.Tab("Listen & Assemble"):
                    gr.Markdown(
                        """
                        ### Saved audio library

                        Choose a saved audiobook job to browse its chunk WAVs, earlier versions,
                        legacy section/master files, and manually assembled WAVs. Select an item
                        to load it into the player. These files stay in the job folder.
                        """
                    )
                    with gr.Row():
                        saved_audiobook_job_input = gr.Dropdown(
                            choices=list_audiobook_job_choices(),
                            label="Saved audiobook job",
                            value=None,
                            interactive=True,
                            scale=5,
                        )
                        refresh_audiobook_jobs_button = gr.Button(
                            "Refresh saved jobs",
                            variant="secondary",
                        )
                    with gr.Row():
                        with gr.Column(scale=1):
                            saved_audiobook_audio_input = gr.Dropdown(
                                choices=[],
                                label="Previously generated WAV",
                                value=None,
                                interactive=True,
                            )
                            saved_audiobook_audio_player = gr.Audio(
                                label="Listen to selected saved WAV",
                                type="filepath",
                            )
                            saved_audiobook_audio_status = gr.Textbox(
                                label="Playback status",
                                lines=2,
                                interactive=False,
                            )
                        with gr.Column(scale=1):
                            gr.Markdown("#### Assemble a master WAV")
                            assembly_chunk_selection_input = gr.Dropdown(
                                choices=[],
                                value=[],
                                multiselect=True,
                                label="Choose chunk WAVs to include",
                                interactive=True,
                            )
                            assembly_all_chunks_input = gr.Checkbox(
                                value=False,
                                label="Use all completed chunks in this job",
                            )
                            assembly_output_name_input = gr.Textbox(
                                label="Optional master filename prefix",
                                placeholder="Leave blank for the saved job name",
                            )
                            assembly_button = gr.Button(
                                "Assemble selected WAVs",
                                variant="primary",
                            )
                            assembly_status_output = gr.Textbox(
                                label="Assembly status",
                                lines=4,
                                interactive=False,
                            )
                            assembled_audio_player = gr.Audio(
                                label="Listen to assembled master",
                                type="filepath",
                            )
                            assembled_master_file_output = gr.File(
                                label="Download assembled master WAV",
                            )
                            assembly_bundle_output = gr.File(
                                label="Updated delivery bundle ZIP",
                            )
                    gr.Markdown(
                        "Selection order in the list does not change the reading order. WAVs are "
                        "assembled in their original manuscript order, with the saved pauses "
                        "between the selected chunks. 'All completed chunks' ignores unfinished "
                        "chunks and includes every valid completed chunk."
                    )
                    refresh_audiobook_jobs_button.click(
                        fn=refresh_audiobook_job_choices,
                        inputs=[],
                        outputs=[
                            saved_audiobook_job_input,
                            saved_audiobook_audio_input,
                            assembly_chunk_selection_input,
                        ],
                        api_name=False,
                    )
                    saved_audiobook_job_input.change(
                        fn=refresh_audiobook_audio_choices,
                        inputs=[saved_audiobook_job_input],
                        outputs=[
                            saved_audiobook_audio_input,
                            assembly_chunk_selection_input,
                        ],
                        api_name=False,
                    )
                    saved_audiobook_audio_input.change(
                        fn=play_saved_audiobook_audio,
                        inputs=[
                            saved_audiobook_job_input,
                            saved_audiobook_audio_input,
                        ],
                        outputs=[
                            saved_audiobook_audio_player,
                            saved_audiobook_audio_status,
                        ],
                        api_name=False,
                    )
                    assembly_button.click(
                        fn=assemble_audiobook_wavs,
                        inputs=[
                            saved_audiobook_job_input,
                            assembly_chunk_selection_input,
                            assembly_all_chunks_input,
                            assembly_output_name_input,
                        ],
                        outputs=[
                            assembled_audio_player,
                            assembled_master_file_output,
                            assembly_status_output,
                            assembly_bundle_output,
                            saved_audiobook_audio_input,
                        ],
                        api_name=False,
                    )

        with gr.Tab("Accent Check"):
            gr.Markdown(
                """
                ### Spoiler-free regional accent scan

                Scan a saved audiobook chunk, assembly, section, or master without transcribing it. The report
                contains only timestamps, broad English accent categories, the competing category,
                and model scores. It never displays sentence text or book content.

                The first scan downloads the AccentCL checkpoint (about 2.56 GB) and Whisper Large v3.
                A GPU is strongly recommended. To avoid a GPU-memory pile-up, Chatterbox is unloaded
                for the scan and reloaded automatically afterward. On a free Colab GPU, a multi-hour
                scan will still take some time.
                """
            )
            accent_audio_input = gr.Dropdown(
                choices=list_saved_accent_audio_choices(),
                label="Saved audiobook chunk, assembly, section, or master",
                value=None,
                interactive=True,
            )
            refresh_accent_audio_button = gr.Button("Refresh saved audio list")
            uploaded_accent_audio_input = gr.File(
                label="Or upload an audio file",
                file_count="single",
                file_types=[".wav", ".mp3"],
                type="filepath",
            )
            accent_analyse_button = gr.Button(
                "Analyze accent without transcription",
                variant="primary",
            )
            accent_summary_output = gr.Markdown()
            accent_segments_output = gr.Dataframe(
                headers=[
                    "Start", "End", "Leading accent", "Score",
                    "Runner-up", "Runner-up score", "Margin",
                ],
                datatype=["str", "str", "str", "str", "str", "str", "str"],
                label="Accent estimates by audio window",
                interactive=False,
                wrap=True,
            )
            with gr.Row():
                accent_json_output = gr.File(label="Download detailed JSON report")
                accent_csv_output = gr.File(label="Download segment CSV")
            refresh_accent_audio_button.click(
                fn=refresh_accent_audio_choices,
                inputs=[],
                outputs=[accent_audio_input],
                api_name=False,
            )
            accent_analyse_button.click(
                fn=run_accent_analysis,
                inputs=[accent_audio_input, uploaded_accent_audio_input],
                outputs=[
                    accent_summary_output,
                    accent_segments_output,
                    accent_json_output,
                    accent_csv_output,
                ],
                api_name=False,
            )

        with gr.Tab("Logs and Recovery"):
            gr.Markdown(
                """
                ### Activity, logs, and recovery records

                Enter an audiobook Job ID to inspect its log and manifest. Leave it blank to
                inspect the quick-TTS app activity log. The job log records chunk starts,
                completions, failures, seeds, elapsed times, and output hashes. It does not
                record your narrator audio. The job manifest contains the manuscript text in
                chunk records, so keep it private.
                """
            )
            saved_jobs_button = gr.Button("List saved audiobook jobs")
            saved_jobs_output = gr.Textbox(
                label="Saved jobs (copy a Job ID to resume)",
                lines=8,
                max_lines=14,
                interactive=False,
            )
            saved_jobs_button.click(
                fn=list_recent_audiobook_jobs,
                inputs=[],
                outputs=[saved_jobs_output],
                api_name=False,
            )
            logs_job_id_input = gr.Textbox(
                label="Audiobook Job ID (blank = quick-TTS activity log)",
                placeholder="Paste the Job ID shown in the Audiobook Jobs tab",
            )
            refresh_logs_button = gr.Button("Refresh logs")
            activity_log_output = gr.Textbox(
                label="Recent activity",
                lines=18,
                max_lines=28,
                interactive=False,
            )
            with gr.Row():
                inspected_manifest_output = gr.File(label="Download inspected manifest")
                inspected_log_output = gr.File(label="Download inspected log")
                inspected_qa_output = gr.File(label="Download technical QA report")
            refresh_logs_button.click(
                fn=load_activity_logs,
                inputs=[logs_job_id_input],
                outputs=[activity_log_output, inspected_manifest_output, inspected_log_output, inspected_qa_output],
                api_name=False,
            )


demo.queue()
print("Starting the Gradio app without login authentication...")
print("After the next cell output appears, open the Gradio public URL.")
demo.launch(
    share=True,
    server_name="0.0.0.0",
    debug=True,
    show_error=True,
    max_file_size="2gb",
    allowed_paths=[str(LOCAL_OUTPUT_DIR)] + ([str(DRIVE_OUTPUT_DIR)] if DRIVE_OUTPUT_DIR is not None else []),
    enable_monitoring=False,
)

## Usage notes and troubleshooting

- **Google Drive is optional:** the notebook does not request Drive access when `USE_GOOGLE_DRIVE = False`. This is the default. Generated WAVs and JSON metadata stay in the Colab VM and can be downloaded through Gradio, but are deleted when the runtime is recycled.
- **Enable Drive persistence:** set `USE_GOOGLE_DRIVE = True` in the setup cell before running it, then execute the notebook from the top. Authorize Google Drive when prompted. The V3 model snapshot and generated WAV/JSON files are then persisted under `MyDrive/Chatterbox Playground`.
- **Model download:** without Drive, the model is downloaded to the Colab VM each time you start a fresh runtime. With Drive enabled, the notebook reuses its verified model snapshot when available.
- **No GPU:** the notebook loads on CPU if CUDA is unavailable, but synthesis will be considerably slower.
- **Gradio link:** the share link is temporary and stops working when the Colab runtime disconnects. Relaunch the final cell to create a new one.
- **Public Gradio link:** authentication is disabled. Anyone with the temporary share URL can use the app. Keep the URL private and stop the Gradio server when finished.
- **Mixed scripts / repeated phrases:** the selected language ID is applied to every character in the input. For English narration, do not insert an isolated Devanagari word unless the English tokenizer actually supports it. The app now records unknown `[UNK]` text tokens in the JSON metadata and warns in the status field. If a comma-separated list loops, try semicolons between items and rerender with a fixed seed; this changes pauses, not the words.
- **Voice reference:** use one speaker, clean speech, little background noise, and preferably 8–10 seconds in the standard profile. The standard profile uses 6 seconds for the T3 speech prompt and 10 seconds for S3Gen. The longer VoiceEncoder-sample profile can feed up to 30 seconds to the T3 VoiceEncoder identity embedding. The extended prompt profile experiments with 10 seconds for T3 and 15 seconds for S3Gen. Those longer prompt windows are not the upstream defaults and should be judged by listening; more audio is not automatically better. Your original upload and normalized temporary copy stay in the active Colab runtime and are not copied to Drive.
- **Repeated phrases / list loops:** in one reproduced case, Temperature **0.55** and Repetition penalty **1.35 together** stopped the repetition; either setting alone did not. Try the pair together while leaving the other controls unchanged. This is an empirically useful preset for repetition-prone passages, not a universal guarantee. The app keeps its general-purpose defaults unchanged.
- **PDF manuscripts:** the Audiobook Jobs tab accepts text-based PDFs. It extracts selectable text page by page, reports pages with little/no extracted text, normalizes physical line wraps, saves the original PDF, and includes it in the delivery ZIP. Scanned/image-only PDFs are flagged; this notebook does not run OCR. OCR them separately and upload the searchable PDF or resulting text.
- **Line-wrap cleanup:** single physical line breaks inside paragraphs become spaces, punctuation is retained, blank lines remain paragraph boundaries, and recognized standalone headings stay separate. The same cleanup is applied to pasted text, TXT/Markdown uploads, PDFs, and Quick TTS. Preview the section/chunk plan first. The job saves the original source, one WAV per chunk, a manifest, logs, per-section masters, a streamed whole-book master, and a delivery ZIP. It checkpoints after each chunk and resumes only when the source text, chunk plan, voice reference, and generation settings match.
- **Resume after Colab reset:** keep USE_GOOGLE_DRIVE = True before running setup. Reopen the notebook, run setup/model cells, upload the exact same narrator reference, enter the original Job ID, restore the same chunk/pause/generation controls, check the approval box, and run again. Completed WAVs are hash-verified and skipped. Do not change settings mid-job; create a new Job ID for a deliberate new render.
- **Seeds in batch jobs:** with base seed -1, the notebook assigns and saves an independent random seed per chunk. Those seeds are reused on resume. With a non-negative base seed, each chunk gets base seed + its zero-based chunk offset. The manifest records every seed.
- **Chunk text and punctuation:** the original manuscript is never edited. When an overlong sentence must be split mid-sentence, the job records a TTS-only continuation comma so Chatterbox's normalizer is less likely to turn that artificial boundary into a full stop.
- **Section selection:** Markdown headings and common headings such as Chapter, Preface, Introduction, Notes, and Appendix start selectable sections. Preview first, then select one or more detected sections to speak; an empty selection means all sections. For a text file with no detected headings, the whole text is one section. Headings are spoken by default; inspect the preview and toggle this before synthesis. The original source file is retained unchanged in the job folder and ZIP.
- **Audio masters:** section WAVs and the whole-book master are 24 kHz mono PCM16. Assembly streams from chunk files instead of loading the complete book into memory. The ZIP includes chunk audio, masters, source, manifest, and logs. Keep it private; the manifest contains your manuscript text.
- **Logs and recovery:** use Logs and Recovery to inspect the quick-TTS event history or a specific audiobook Job ID. The live log updates after each chunk and is saved both as readable text and JSONL events. The same tab can list saved jobs after a runtime reset and download a job's manifest, log, and QA report.
- **Tokenizer audit:** batch jobs count unknown [UNK] text tokens per chunk and record the results in the manifest/QA report. A nonzero count is a review warning, not an automatic rewrite or a reason to silently drop text.
- **Fix a bad portion:** in Create / Resume, enter the existing Job ID and click **Load chunks from this Job ID**. The multiselect lists each chunk in manuscript order with its ID, saved status, section, and a short text excerpt. Select only the problematic chunk (for example C00007) or a section ID (for example S002), then run Generate / resume with the same manuscript, narrator/style references, and generation settings. Only the selected chunks are regenerated; all other verified chunk WAVs are skipped. A successful rerender atomically replaces the active WAV. Enable **Keep replaced WAVs as backups** to retain the prior version under revisions/ and in the delivery ZIP. If synthesis fails before replacement, the old valid WAV stays active.
- **Technical QA report:** the job writes qa_report.json and includes it in the delivery ZIP. It checks WAV hashes/format, unusual duration or estimated speaking rate, sentence-internal splits, and limiter engagement. Flags are listening prompts only; the report cannot detect accent, pronunciation, repetition, prosody, or omitted meaning.
- **Full-book safeguard:** the per-job checkbox is a user confirmation only. It does not replace manuscript approval or unlock the separate document-first audiobook workflow's approval gate.
- **Long passages in Quick TTS:** Quick TTS still uses 420-character chunks and is intended for auditions, not whole-book processing.
- **Output privacy:** quick-TTS metadata includes the entered text and settings. Audiobook job manifests and bundles include the manuscript text, settings, seeds, logs, and audio. Keep the Drive folder or downloaded files private.
- **Model mismatch:** if the V3 selector is missing or the notebook imports an old package, restart the Colab runtime and rerun from the top. The notebook intentionally uses the pinned upstream source commit rather than the old PyPI wheel.
- **This is not book synthesis:** nothing in this notebook changes the manuscript, pronunciation decisions, roadmap, or full-book synthesis lock.

Useful links: [Chatterbox source](https://github.com/resemble-ai/chatterbox) · [Gradio Audio component](https://gradio.app/docs/audio) · [Gradio Blocks](https://gradio.app/docs/gradio/blocks).

### Regional accent analysis (optional)

The Accent Check tab runs the open-source AccentCL English accent classifier against a saved audiobook chunk, assembly, section, or master WAV, or an uploaded WAV/MP3. It scans overlapping 15-second windows with a 10-second hop, decodes audio incrementally with FFmpeg, and reports timestamps, the leading regional category, the runner-up, and the model's softmax scores. It never runs speech recognition or includes manuscript sentences in its output.

The released model's labels are broad groups: North American, British Isles, Australasian, South Asian, Southeast Asian, Spanish-accented English, and Chinese-accented English. It does not identify fine distinctions such as RP versus Estuary. A sustained label change is only highlighted when the same new category is predicted for at least two windows.

**Model cost and limitations:** AccentCL downloads a roughly 2.56 GB classifier checkpoint plus Whisper Large v3 on first use. The notebook caches both the classifier checkpoint and Whisper backbone under the selected output storage, so enabling Drive persistence keeps both downloads across runtime resets. A GPU is strongly recommended. To avoid holding Chatterbox and Whisper in VRAM at once, the scan temporarily unloads Chatterbox and reloads it automatically afterward. CPU inference is supported but can be very slow.

The score is not a calibrated probability that a label is correct. AccentCL was evaluated on human English speech, so its published benchmark scores are not a guarantee on synthetic Chatterbox narration. Use it to find passages worth checking, especially sustained changes between North American and British Isles categories. It does not assess pronunciation of Sanskrit/Buddhist terms, pacing, clipping, or sentence-level correctness. The final judgement of a narrator's consistency still requires spot-checking audio.

Reports are saved as private JSON and CSV files. They contain audio file name/hash, timestamped labels, scores, and broad regions, but no transcript and no audio copy. Keep the reports private alongside the audiobook outputs.


- **Combined WAV output:** when every planned section is complete, the notebook automatically stitches the active chunk WAVs into individual section/chapter WAVs and one combined WAV. Full-source jobs produce whole_book_master.wav; jobs restricted to selected sections produce selected_sections_master.wav. The final combined master is included in the delivery ZIP. Rerendering rebuilds these masters so they contain the latest active chunk audio.
